# Tree growth: Most simple regression model

In this notebook, we will make the simplest possible regression model for the project, modeling growth based only on previous tree height and moisture level. 

<img src="http://mlsm.man.dtu.dk/mbml/wall_street.png">

The relevant imports.

In [49]:
import numpy as np
import pandas as pd   # We import Pandas!
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn import linear_model
import torch
import itertools

import libpysal
from esda.moran import Moran
from sklearn.metrics import r2_score

import pyro
import pyro.distributions as dist
from pyro.contrib.autoguide import AutoDiagonalNormal, AutoMultivariateNormal
from pyro.infer import MCMC, NUTS, HMC, SVI, Trace_ELBO
from pyro.optim import Adam, ClippedAdam

# fix random generator seed (for reproducibility of results)
np.random.seed(42)

# matplotlib options
palette = itertools.cycle(sns.color_palette())
plt.style.use('ggplot')
%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 10)

Lets start by loading the dataset. In order to let you focus on the probabilistic modelling aspects, we already prepared the raw GPS taxi data for you and extended it with additional information about the weather conditions.

In [50]:
# load csv (original dataset is by 30min intervals, we want 1h intervals) into a Pandas Dataframe
# df = pd.read_csv("https://mlsm.man.dtu.dk/mbml/pickups+weather_wallstreet.csv")

# look at the first few lines of the loaded dataset
# df.head()
df = pd.read_csv(
    "C:\\Users\\rasmu\\OneDrive - Danmarks Tekniske Universitet\\Dokumenter\\MBML project\\data\\out_10km_idx_preprocessed.csv",
    nrows=5000,          # start with a subset
    low_memory=False
)

df.head()


,x,y,biomassa_omdrev1,biomassa_omdrev2,flodesackumulering,grundyta_omdrev1,grundyta_omdrev2,markfuktighet,markfuktighet_klassad,medeldiameter_omdrev1,...,CenterLanNamn,CenterKommunNamn,is_no_forest,is_lake,delta_neg_medelhojd,delta_neg_p95,delta_neg_medeldiameter,delta_neg_biomassa,delta_neg_volym,is_stable_forest
0,445881.25,6260468.75,79.0,113.0,0.018730,22.0,23.0,97.0,2.0,15.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True
1,445893.75,6260468.75,65.0,96.0,0.121874,17.0,20.0,99.0,2.0,18.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True
2,445906.25,6260468.75,76.0,117.0,0.014839,21.0,24.0,99.0,2.0,15.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True
3,445918.75,6260468.75,68.0,113.0,0.004559,19.0,23.0,85.0,2.0,14.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True
4,445931.25,6260468.75,76.0,136.0,0.001676,21.0,27.0,82.0,2.0,14.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True


Filtering on "Is_stable_forest" 

In [51]:
# Keep only stable-forest rows
col_lookup = {c.lower(): c for c in df.columns}
stable_col = col_lookup.get("is_stable_forest")

# Fallback: find a likely stable-forest column if naming differs.
if stable_col is None:
    candidates = [
        c for c in df.columns
        if ("stable" in c.lower()) and ("forest" in c.lower())
    ]
    stable_col = candidates[0] if candidates else None

if stable_col is None:
    raise KeyError("No stable-forest column found (expected something like 'Is_stable_forest').")

stable_raw = df[stable_col]
if pd.api.types.is_bool_dtype(stable_raw):
    stable_mask = stable_raw
else:
    stable_mask = stable_raw.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])

df = df.loc[stable_mask].copy()
print(f"Using stable-forest column: {stable_col}")
print(f"Rows after stable-forest filter: {len(df)}")
df.head()

Using stable-forest column: is_stable_forest
Rows after stable-forest filter: 2003


,x,y,biomassa_omdrev1,biomassa_omdrev2,flodesackumulering,grundyta_omdrev1,grundyta_omdrev2,markfuktighet,markfuktighet_klassad,medeldiameter_omdrev1,...,CenterLanNamn,CenterKommunNamn,is_no_forest,is_lake,delta_neg_medelhojd,delta_neg_p95,delta_neg_medeldiameter,delta_neg_biomassa,delta_neg_volym,is_stable_forest
0,445881.25,6260468.75,79.0,113.0,0.018730,22.0,23.0,97.0,2.0,15.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True
1,445893.75,6260468.75,65.0,96.0,0.121874,17.0,20.0,99.0,2.0,18.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True
2,445906.25,6260468.75,76.0,117.0,0.014839,21.0,24.0,99.0,2.0,15.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True
3,445918.75,6260468.75,68.0,113.0,0.004559,19.0,23.0,85.0,2.0,14.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True
4,445931.25,6260468.75,76.0,136.0,0.001676,21.0,27.0,82.0,2.0,14.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True


The model considers growth as a function of various forest characteristics including tree dimensions, volume, biomass, and soil wetness.



In [52]:
# prepare matrix with selected domain features (explicit list)
requested_cols = [
    "biomassa_omdrev1",
    "grundyta_omdrev1",
    "medeldiameter_omdrev1",
    "medelhojd_omdrev1",
    "p95_omdrev1",
    "vegetationskvot_omdrev1",
    "volym_omdrev1",
    "flodesackumulering",
    "markfuktighet",
]

available_cols = [c for c in requested_cols if c in df.columns]
missing_cols = [c for c in requested_cols if c not in df.columns]

if not available_cols:
    X_selected = np.empty((len(df), 0))
    X_weather = X_selected
    print("None of the requested columns were found in df.")
else:
    # Convert selected columns to numeric and handle missing values.
    X_selected = (
        df[available_cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
        .values
    )

    # Keep existing downstream code compatible.
    X_weather = X_selected

    print(f"Using {len(available_cols)} requested columns")
    print(available_cols)
    if missing_cols:
        print(f"Missing {len(missing_cols)} requested columns")
        print(missing_cols)
    print(X_weather.shape)

Using 9 requested columns
['biomassa_omdrev1', 'grundyta_omdrev1', 'medeldiameter_omdrev1', 'medelhojd_omdrev1', 'p95_omdrev1', 'vegetationskvot_omdrev1', 'volym_omdrev1', 'flodesackumulering', 'markfuktighet']
(2003, 9)


In [53]:
def preprocess_species_clr(df, species_cols=None, eps=1e-9):
    if species_cols is None:
        species_cols = [
            "slu_skogskarta_biomassa",
            "slu_skogskarta_bjork_volym",
            "slu_skogskarta_bok_volym",
            "slu_skogskarta_contorta_volym",
            "slu_skogskarta_ek_volym",
            "slu_skogskarta_gran_volym",
            "slu_skogskarta_ovrigt_volym",
            "slu_skogskarta_tall_volym",
        ]

    present = [c for c in species_cols if c in df.columns]
    n_rows = len(df)
    n_sp = len(species_cols)

    if not present:
        perc = np.zeros((n_rows, n_sp), dtype=float)
        clr = np.zeros_like(perc)
        names = [c + "_clr" for c in species_cols]
        return perc, clr, names

    vals = df[present].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)
    row_sums = vals.sum(axis=1)

    # Percentages (0-100)
    perc = np.zeros_like(vals)
    nz = row_sums > 0
    if nz.any():
        perc[nz] = (vals[nz] / row_sums[nz][:, None]) * 100.0

    # CLR on proportions (use eps for zeros)
    clr = np.zeros_like(vals)
    for i in range(n_rows):
        if row_sums[i] <= 0:
            continue
        props = vals[i] / row_sums[i]
        props_safe = np.where(props <= 0, eps, props)
        gmean = np.exp(np.log(props_safe).mean())
        clr[i] = np.log(props_safe / gmean)

    # Reorder to full species_cols order and pad missing columns with zeros
    if len(present) != n_sp:
        perc_full = np.zeros((n_rows, n_sp), dtype=float)
        clr_full = np.zeros((n_rows, n_sp), dtype=float)
        for j, c in enumerate(species_cols):
            if c in present:
                idx = present.index(c)
                perc_full[:, j] = perc[:, idx]
                clr_full[:, j] = clr[:, idx]
        perc = perc_full
        clr = clr_full
        names = [c + "_clr" for c in species_cols]
    else:
        names = [c + "_clr" for c in present]

    return perc, clr, names

# Example usage (uncomment to run):
# perc, clr, clr_names = preprocess_species_clr(df)
# print('CLR shape:', clr.shape, 'CLR names:', clr_names)


In [54]:
# Integrate species CLR features into the feature matrix
perc, clr, clr_names = preprocess_species_clr(df)
print(f"Computed CLR matrix with shape: {clr.shape}")

# Append CLR columns to X_weather
if clr.size > 0:
    # Ensure X_weather is 2D numpy array
    if X_weather is None:
        X_weather = clr
    else:
        X_weather = np.hstack([X_weather, clr])
    print(f"Appended {clr.shape[1]} CLR features: {clr_names}")
else:
    print("No CLR features found or all species columns missing; nothing appended.")

# Keep a list of feature names (existing requested ones + CLR names) for reference
try:
    feature_names = available_cols + clr_names
except NameError:
    feature_names = clr_names

print(f"New X_weather shape: {X_weather.shape}")


Computed CLR matrix with shape: (2003, 8)
Appended 8 CLR features: ['slu_skogskarta_biomassa_clr', 'slu_skogskarta_bjork_volym_clr', 'slu_skogskarta_bok_volym_clr', 'slu_skogskarta_contorta_volym_clr', 'slu_skogskarta_ek_volym_clr', 'slu_skogskarta_gran_volym_clr', 'slu_skogskarta_ovrigt_volym_clr', 'slu_skogskarta_tall_volym_clr']
New X_weather shape: (2003, 17)


In [55]:
# Register the CLR columns in df so later spatial lag code can reference them directly.
# The CLR values are part of the model features even though they are derived columns.
if 'clr_names' not in globals():
    raise ValueError('clr_names is not available yet; run the CLR preprocessing cell first.')

clr_df = pd.DataFrame(clr, columns=clr_names, index=df.index)
df = pd.concat([df, clr_df], axis=1)

# Use one explicit feature-name list for all downstream model variants.
model_feature_cols = [c for c in available_cols + clr_names if c in df.columns]
print(f"Registered CLR columns in df: {clr_names}")
print(f"Model feature columns ({len(model_feature_cols)}): {model_feature_cols}")


Registered CLR columns in df: ['slu_skogskarta_biomassa_clr', 'slu_skogskarta_bjork_volym_clr', 'slu_skogskarta_bok_volym_clr', 'slu_skogskarta_contorta_volym_clr', 'slu_skogskarta_ek_volym_clr', 'slu_skogskarta_gran_volym_clr', 'slu_skogskarta_ovrigt_volym_clr', 'slu_skogskarta_tall_volym_clr']
Model feature columns (17): ['biomassa_omdrev1', 'grundyta_omdrev1', 'medeldiameter_omdrev1', 'medelhojd_omdrev1', 'p95_omdrev1', 'vegetationskvot_omdrev1', 'volym_omdrev1', 'flodesackumulering', 'markfuktighet', 'slu_skogskarta_biomassa_clr', 'slu_skogskarta_bjork_volym_clr', 'slu_skogskarta_bok_volym_clr', 'slu_skogskarta_contorta_volym_clr', 'slu_skogskarta_ek_volym_clr', 'slu_skogskarta_gran_volym_clr', 'slu_skogskarta_ovrigt_volym_clr', 'slu_skogskarta_tall_volym_clr']


In [56]:
# prepare target as growth in mean height between omdrev2 and omdrev1
y_raw = (
    pd.to_numeric(df["medelhojd_omdrev2"], errors="coerce")
    - pd.to_numeric(df["medelhojd_omdrev1"], errors="coerce")
)

# Fill any missing values before standardization.
y = y_raw.fillna(0.0).values

# standardize target
y_mean = y.mean()
y_std = y.std()
y = (y - y_mean) / y_std
print(y.shape)
print(f"Target mean (before standardization): {y_mean:.4f}")
print(f"Target std (before standardization): {y_std:.4f}")

(2003,)
Target mean (before standardization): 35.1907
Target std (before standardization): 27.1310


The X matrix now contain all the input data for the model, and the y vector contains all the corresponding targets (growth).

The next step is to split our data into a train and test set. Alternatively, we could have used something like cross-validation, but for the sake of simplicity, a train/test split will do just fine for this example.

In [57]:
train_perc = 0.66  # percentage of training data
split_point = int(train_perc * len(y))
perm = np.random.permutation(len(y))
ix_train = perm[:split_point]
ix_test = perm[split_point:]

# Use the selected feature matrix as model input.
X = X_weather
X_train = X[ix_train, :]
X_test = X[ix_test, :]
y_train = y[ix_train]
y_test = y[ix_test]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print("num train: %d" % len(y_train))
print("num test: %d" % len(y_test))

# ADD THIS:
# Standardize input features
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0) + 1e-8  # avoid division by zero
X_train = (X_train - X_mean) / X_std
X_test = (X_test - X_mean) / X_std
print("\nFeatures standardized:")
print(f"X_train mean: {X_train.mean(axis=0)}")
print(f"X_train std: {X_train.std(axis=0)}")

print(f"y_train mean: {y_train.mean():.4f}")
print(f"y_train std: {y_train.std():.4f}")

X_train shape: (1321, 17)
X_test shape: (682, 17)
num train: 1321
num test: 682

Features standardized:
X_train mean: [-1.50439002e-17 -1.46236795e-17 -8.64814150e-17 -6.38735427e-18
  1.41530324e-16  2.09269896e-17  3.66852650e-17 -9.42344865e-18
  4.53838330e-17  1.37832382e-17  1.35445528e-15 -2.75244542e-17
  1.31579498e-15  9.58943581e-16  2.80808264e-15  6.26885204e-16
 -1.05786353e-15]
X_train std: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
y_train mean: -0.0250
y_train std: 0.9297


## Split Audit and Spatially Separated Train/Test Split

This section does two things:

1. Audits the current split for index leakage and spatial proximity leakage.
2. Builds a spatially separated split (block holdout + distance buffer) so test locations are far from train locations.

Use the resulting `ix_train_active` and `ix_test_active` in spatial models when you want conservative, leakage-resistant evaluation.


In [58]:
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

# ----- helpers -----
def split_separation_report(train_idx, test_idx, coords, name="split"):
    train_idx = np.asarray(train_idx, dtype=int)
    test_idx = np.asarray(test_idx, dtype=int)

    overlap = np.intersect1d(train_idx, test_idx)
    print(f"\n{name} summary")
    print("-" * 60)
    print(f"Train size: {len(train_idx)}")
    print(f"Test size : {len(test_idx)}")
    print(f"Overlap   : {len(overlap)}")

    if len(overlap) > 0:
        print("WARNING: Train/test overlap found.")
        return

    train_coords = coords[train_idx]
    test_coords = coords[test_idx]

    nn = NearestNeighbors(n_neighbors=1)
    nn.fit(train_coords)
    d_test_to_train, _ = nn.kneighbors(test_coords)
    d_test_to_train = d_test_to_train.ravel()

    print("Nearest train distance for each test point:")
    print(f"  min   : {np.min(d_test_to_train):.4f}")
    print(f"  p10   : {np.percentile(d_test_to_train, 10):.4f}")
    print(f"  p50   : {np.percentile(d_test_to_train, 50):.4f}")
    print(f"  p90   : {np.percentile(d_test_to_train, 90):.4f}")
    print(f"  max   : {np.max(d_test_to_train):.4f}")


def make_spatial_block_split(
    coords,
    target_test_frac=0.34,
    n_blocks=12,
    buffer_mult=3.0,
    random_state=42,
):
    """
    Create a conservative spatial split:
    1) Hold out one spatial KMeans block as test region.
    2) Remove nearby training points within a buffer distance from test region.
    """
    n = coords.shape[0]

    # Block partition of space
    n_blocks = min(max(4, n_blocks), max(4, n - 1))
    km = KMeans(n_clusters=n_blocks, random_state=random_state, n_init=10)
    block_labels = km.fit_predict(coords)

    # Pick the block whose size is closest to the desired test fraction
    block_sizes = np.bincount(block_labels)
    target_test_size = int(round(target_test_frac * n))
    test_block = int(np.argmin(np.abs(block_sizes - target_test_size)))

    ix_test_spatial = np.where(block_labels == test_block)[0]
    ix_train_candidate = np.where(block_labels != test_block)[0]

    # Characteristic local distance from candidate train points
    nn_local = NearestNeighbors(n_neighbors=2)
    nn_local.fit(coords[ix_train_candidate])
    d_local, _ = nn_local.kneighbors(coords[ix_train_candidate])
    typical_nn = float(np.median(d_local[:, 1]))
    buffer_distance = float(buffer_mult * typical_nn)

    # Apply buffer: drop train points too close to any test point
    nn_to_test = NearestNeighbors(n_neighbors=1)
    nn_to_test.fit(coords[ix_test_spatial])
    d_to_test, _ = nn_to_test.kneighbors(coords[ix_train_candidate])
    d_to_test = d_to_test.ravel()

    keep_mask = d_to_test >= buffer_distance
    ix_train_spatial = ix_train_candidate[keep_mask]
    ix_buffer_drop = ix_train_candidate[~keep_mask]

    info = {
        "test_block": test_block,
        "n_blocks": n_blocks,
        "target_test_size": target_test_size,
        "buffer_distance": buffer_distance,
        "typical_nn": typical_nn,
        "n_buffer_drop": int(len(ix_buffer_drop)),
    }
    return ix_train_spatial, ix_test_spatial, ix_buffer_drop, info


# ----- run audit + build spatial split -----
if "x" not in df.columns or "y" not in df.columns:
    raise KeyError("Need x and y columns in df to audit/build a spatial split.")

coords_all = (
    df[["x", "y"]]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
    .to_numpy(dtype=float)
)

print("Current random split audit (ix_train/ix_test):")
split_separation_report(ix_train, ix_test, coords_all, name="Current split")

ix_train_spatial, ix_test_spatial, ix_buffer_drop, split_info = make_spatial_block_split(
    coords_all,
    target_test_frac=len(ix_test) / len(df),
    n_blocks=12,
    buffer_mult=3.0,
    random_state=42,
)

print("\nSpatial split construction details")
print("-" * 60)
for k, v in split_info.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")
print(f"Train points dropped by buffer: {len(ix_buffer_drop)}")

split_separation_report(ix_train_spatial, ix_test_spatial, coords_all, name="Spatial block+buffer split")

# Active split to use in downstream models when avoiding spatial leakage
ix_train_active = ix_train_spatial
ix_test_active = ix_test_spatial

print("\nActive split set:")
print(f"ix_train_active: {len(ix_train_active)}")
print(f"ix_test_active : {len(ix_test_active)}")
print("Use ix_train_active/ix_test_active in spatial model cells for conservative evaluation.")


Current random split audit (ix_train/ix_test):

Current split summary
------------------------------------------------------------
Train size: 1321
Test size : 682
Overlap   : 0
Nearest train distance for each test point:
  min   : 12.5000
  p10   : 12.5000
  p50   : 12.5000
  p90   : 17.6777
  max   : 45.0694

Spatial split construction details
------------------------------------------------------------
test_block: 1
n_blocks: 12
target_test_size: 682
buffer_distance: 37.5000
typical_nn: 12.5000
n_buffer_drop: 1
Train points dropped by buffer: 1

Spatial block+buffer split summary
------------------------------------------------------------
Train size: 1748
Test size : 254
Overlap   : 0
Nearest train distance for each test point:
  min   : 37.5000
  p10   : 150.0000
  p50   : 275.1420
  p90   : 415.5193
  max   : 450.6939

Active split set:
ix_train_active: 1748
ix_test_active : 254
Use ix_train_active/ix_test_active in spatial model cells for conservative evaluation.


c:\Users\rasmu\miniconda3\envs\mbml\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=8.
  warnings.warn(


A crucial step in developing a experimental setup in machine learning is establishing how to access the quality of the models that we learn. For this purpose, we developed a funciton which already constains a series of popular metrics for evaluating the quality of the predictions of a regression model (continuous output variables!). Here we just reuse the assesment from class, but it might be relevant to consider changing. 

In [59]:
def compute_error(trues, predicted):
    corr = np.corrcoef(predicted, trues)[0,1]
    mae = np.mean(np.abs(predicted - trues))
    rae = np.sum(np.abs(predicted - trues)) / np.sum(np.abs(trues - np.mean(trues)))
    rmse = np.sqrt(np.mean((predicted - trues)**2))
    r2 = max(0, 1 - np.sum((trues-predicted)**2) / np.sum((trues - np.mean(trues))**2))
    return corr, mae, rae, rmse, r2

We add some further metrics. 

In [60]:
def evaluate_model(
    y_true,
    y_pred,
    coords=None,
    w=None,
    eps=1e-8,
):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    residuals = y_pred - y_true

    metrics = {
        "MAE": np.mean(np.abs(residuals)),
        "RMSE": np.sqrt(np.mean(residuals**2)),
        "Bias": np.mean(residuals),
        "sMAPE": np.mean(2 * np.abs(residuals) / (np.abs(y_true) + np.abs(y_pred) + eps)),
        "R2": r2_score(y_true, y_pred),
        "Correlation": np.corrcoef(y_true, y_pred)[0, 1]
    }

    if w is not None:
        mi = Moran(residuals, w)
        metrics["Moran_I"] = mi.I
        metrics["Moran_p"] = mi.p_sim
    elif coords is not None:
        coords = np.asarray(coords)
        if coords.ndim != 2 or coords.shape[1] != 2:
            raise ValueError(f"coords must be a 2D array with shape (n_samples, 2); got {coords.shape}")
        if coords.shape[0] != len(y_true):
            raise ValueError(f"coords must have the same number of rows as y_true/y_pred (got {coords.shape[0]} vs {len(y_true)})")

        from libpysal.weights import KNN
        from esda.moran import Moran as MoranLocal

        w_local = KNN.from_array(coords, k=8)
        w_local.transform = "r"
        mi = MoranLocal(residuals, w_local)

        metrics["Moran_I"] = mi.I
        metrics["Moran_p"] = mi.p_sim

    return metrics

I just added some data inspection for debugging and to see what's going on. The data has been standardized. 

In [61]:
# Inspect data quality
print("="*50)
print("DATA QUALITY CHECK")
print("="*50)
print("\nX_train stats:")
print(f"  Mean: {X_train.mean(axis=0)}")
print(f"  Std: {X_train.std(axis=0)}")
print(f"  Min: {X_train.min(axis=0)}")
print(f"  Max: {X_train.max(axis=0)}")

print("\ny_train stats:")
print(f"  Mean: {y_train.mean():.4f}, Std: {y_train.std():.4f}")

print("\nCorrelation with target:")
for i, col in enumerate(available_cols):
    corr = np.corrcoef(X_train[:, i], y_train)[0, 1]
    print(f"  {col}: {corr:.4f}")
print("="*50 + "\n")

DATA QUALITY CHECK

X_train stats:
  Mean: [-1.50439002e-17 -1.46236795e-17 -8.64814150e-17 -6.38735427e-18
  1.41530324e-16  2.09269896e-17  3.66852650e-17 -9.42344865e-18
  4.53838330e-17  1.37832382e-17  1.35445528e-15 -2.75244542e-17
  1.31579498e-15  9.58943581e-16  2.80808264e-15  6.26885204e-16
 -1.05786353e-15]
  Std: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
  Min: [-1.20367851 -1.50740982 -1.53212853 -1.5747366  -1.502849   -1.52249576
 -1.10125904 -0.11864538 -1.46326978 -2.52815147 -3.33448655 -1.34406054
 -2.0791301  -1.3964488  -5.42695207 -1.40037798 -2.91898236]
  Max: [ 4.32142027  2.88169562  2.88869084  2.36016645  2.47859016  1.491647
  4.53756573 19.40480901  1.28826312  1.72469842  1.45566746  4.51527814
  2.5373988   2.69572538  1.74442435  2.62405353  1.57570089]

y_train stats:
  Mean: -0.0250, Std: 0.9297

Correlation with target:
  biomassa_omdrev1: -0.5011
  grundyta_omdrev1: -0.5338
  medeldiameter_omdrev1: -0.5831
  medelhojd_omdrev1: -0.5945
  

For the sake of comparision, we compare to linear regression from the sklearn package.

In [62]:
#regr = linear_model.LinearRegression()
regr = linear_model.Ridge()
regr.fit(X_train, y_train)
y_hat = regr.predict(X_test)

# Convert back to the original scale
preds = y_hat * y_std + y_mean
y_true = y_test * y_std + y_mean

corr, mae, rae, rmse, r2 = compute_error(y_true, preds)
print("CorrCoef: %.3f\nMAE: %.3f\nRMSE: %.3f\nR2: %.3f" % (corr, mae, rmse, r2))

metrics = evaluate_model(y_true, preds)
print("\nNew Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


CorrCoef: 0.548
MAE: 13.550
RMSE: 25.558
R2: 0.295

New Evaluation Metrics:
MAE: 13.5498
RMSE: 25.5579
Bias: -1.9453
sMAPE: 0.3861
R2: 0.2953
Correlation: 0.5477



Better, right? But this took quite longer to run, and in this case didn't necessarily beat "sklearn"...

MCMC methods have great properties, namely the fact that in the limit of infinite computation time they will converge to the true posterior distribution. However, they often have difficulty scaling to larger datasets. 

### Pyro: Train on full dataset using Stochastic Variational Inference (SVI)

SVI on the other hand is much more scalable. Let us now try to use SVI to perform inference in our model. We start by preparing the data by converting it to torch tensors:

In [63]:
# Prepare data for Pyro model
X_train_torch = torch.tensor(X_train).float()
y_train_torch = torch.tensor(y_train).float()

In [64]:
def model(X, obs=None):
    alpha = pyro.sample("alpha", dist.Normal(0., 1.))
    
    beta  = pyro.sample("beta", dist.Normal(torch.zeros(X.shape[1]), 
                                            torch.ones(X.shape[1]) * 10.0).to_event())  # Weaker prior
    sigma = pyro.sample("sigma", dist.HalfCauchy(5.))
    
    with pyro.plate("data"):
        y = pyro.sample("y", dist.Normal(alpha + X.matmul(beta), sigma), obs=obs)
        
    return y

As we briefly touched upon during the notebook from last lecture (and also in the slides), in VI we specify a parametric distribution $q(\textbf{z}|\boldsymbol\phi)$ (called the *variational distribution*) with parameters $\boldsymbol\phi$. The goal is the to find the values of the parameters $\boldsymbol\phi$ that make $q(\textbf{z}|\boldsymbol\phi)$ and close as possible to the true posterior distribution $p(\textbf{z}|\textbf{x})$, thereby turning the problem of Bayesian inference into an optimization problem. We will get back to explain VI in detail later on in the course, but for now this is the essential that you need to know to use it in Pyro. 

In Pyro, the variational distribution is specified in a ```guide```, just like the model function. In fact, there are classes (e.g. ```AutoDiagonalNormal``` and ```AutoMultivariateNormal```) that generate the guide function automatically: 

In [65]:
# Define guide function
guide = AutoMultivariateNormal(model)

# Reset parameter values
pyro.clear_param_store()

Notice that we also reset the storage of Pyro parameters using ```pyro.clear_param_store()```. This is particularly important when running inference on the same model multiple times, because otherwise, Pyro will remember and re-use the parameter values from the last execution.

As mentioned above, in VI, the problem of (approximate) Bayesian inference is turned into an optimization problem. Like any other numerical optimization problem, we need to specify the optimizer (in this case ```ClippedAdam``` - a gradient descent algorithm that cleverly adapts the step size), an objective or loss function (```Trace_ELBO``` - this is the default that you will almost always use), and other details like the learning rate of the optimizer and the number of optimization steps:

In [66]:
# Define the number of optimization steps
n_steps = 4000

# Setup the optimizer
adam_params = {"lr": 0.001} # learning rate (lr) of optimizer
optimizer = ClippedAdam(adam_params)

# Setup the inference algorithm
elbo = Trace_ELBO(num_particles=1)
svi = SVI(model, guide, optimizer, loss=elbo)

The last step above was to instantiate ```SVI``` with the ```model```, ```guide```, ```optimizer``` and loss function (```elbo```) that we have just defined. Once this is done, we can solve this optimization problem by taking steps in the direction of the gradient using the function ```svi.step(...)```:

In [67]:
# Do gradient steps
for step in range(n_steps):
    elbo = svi.step(X_train_torch, y_train_torch)
    if step % 100 == 0:
        print("[%d] ELBO: %.1f" % (step, elbo))

[0] ELBO: 15097.0
[100] ELBO: 14051.4
[200] ELBO: 10762.2
[300] ELBO: 7928.2
[400] ELBO: 7752.0
[500] ELBO: 7099.8
[600] ELBO: 6283.4
[700] ELBO: 5684.1
[800] ELBO: 5469.0
[900] ELBO: 5394.9
[1000] ELBO: 5256.5
[1100] ELBO: 5163.0
[1200] ELBO: 5052.1
[1300] ELBO: 5009.8
[1400] ELBO: 4961.3
[1500] ELBO: 4920.1
[1600] ELBO: 4790.2
[1700] ELBO: 4751.3
[1800] ELBO: 4699.4
[1900] ELBO: 4639.5
[2000] ELBO: 4570.8
[2100] ELBO: 4529.0
[2200] ELBO: 4490.1
[2300] ELBO: 4425.5
[2400] ELBO: 4375.5
[2500] ELBO: 4356.3
[2600] ELBO: 4298.6
[2700] ELBO: 4174.4
[2800] ELBO: 4203.2
[2900] ELBO: 4121.5
[3000] ELBO: 4110.2
[3100] ELBO: 4084.8
[3200] ELBO: 4020.7
[3300] ELBO: 3984.6
[3400] ELBO: 3909.1
[3500] ELBO: 3914.9
[3600] ELBO: 3820.6
[3700] ELBO: 3830.7
[3800] ELBO: 3720.6
[3900] ELBO: 3637.4


If all went well, the values of the loss function in the output above should be going down. They can ocasionally go up, because Pyro estimates stochastic gradients (i.e. a noisy approximation to the gradients that, although imperfect is very efficient and scalable), but overall trend should be for the values of the loss function to go down. 

Once the optimization has converged to a minimum, we can use the ```Predictive``` class to extract the results. The usage is similar to what we did with MCMC, although we now must pass the guide to the ```Predictive``` class as well:

In [68]:
from pyro.infer import Predictive

predictive = Predictive(model, guide=guide, num_samples=1000,
                        return_sites=("alpha", "beta", "sigma"))
samples = predictive(X_train_torch, y_train_torch)

We can now use the samples from the posterior distribution above to make predictions for the test set:

In [69]:
alpha_samples = samples["alpha"].detach().numpy().ravel()  # Force 1D
beta_samples = samples["beta"].detach().numpy()

# Determine number of samples and features
n_samples = len(alpha_samples)
n_features = X_test.shape[1]

# Reshape beta to (n_samples, n_features)
beta_samples = beta_samples.reshape(n_samples, n_features)

# Compute predictions for all samples at once
# X_test is (682, 9), beta_samples is (1000, 9)
# X_test @ beta_samples.T gives (682, 1000)
# Add alpha to each column (broadcasting)
preds_all = X_test @ beta_samples.T + alpha_samples[np.newaxis, :]
# Average across samples
y_hat = preds_all.mean(axis=1)

In [70]:
# Debug: check shapes before prediction
print("Raw alpha_samples shape:", samples["alpha"].detach().numpy().shape)
print("Raw beta_samples shape:", samples["beta"].detach().numpy().shape)
print("X_test shape:", X_test.shape)
print()

# Extract and flatten
alpha_samples = samples["alpha"].detach().numpy().flatten()
beta_samples = samples["beta"].detach().numpy()
print("After flatten - alpha shape:", alpha_samples.shape)
print("After flatten - beta shape:", beta_samples.shape)
print()

# Reshape beta
beta_samples = beta_samples.reshape(1000, -1)
print("After reshape - beta shape:", beta_samples.shape)
print("beta.T shape:", beta_samples.T.shape)

Raw alpha_samples shape: (1000, 1)
Raw beta_samples shape: (1000, 1, 17)
X_test shape: (682, 17)

After flatten - alpha shape: (1000,)
After flatten - beta shape: (1000, 1, 17)

After reshape - beta shape: (1000, 17)
beta.T shape: (17, 1000)


In [71]:
alpha_samples = samples["alpha"].detach().numpy()
beta_samples = samples["beta"].detach().numpy()
y_hat = np.mean(alpha_samples.T + np.dot(X_test, beta_samples[:,0].T), axis=1)

# convert back to the original scale
preds = y_hat * y_std + y_mean
y_true = y_test * y_std + y_mean

corr, mae, rae, rmse, r2 = compute_error(y_true, preds)
print("CorrCoef: %.3f\nMAE: %.3f\nRMSE: %.3f\nR2: %.3f" % (corr, mae, rmse, r2))

metrics = evaluate_model(y_true, preds)
print("\nSVI New Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

CorrCoef: 0.099
MAE: 81.045
RMSE: 101.144
R2: 0.000

SVI New Evaluation Metrics:
MAE: 81.0448
RMSE: 101.1440
Bias: -2.7067
sMAPE: 1.4048
R2: -10.0362
Correlation: 0.0989


In [72]:
# === Exact conjugate Bayesian linear regression (Normal-Inverse-Gamma) ===
# This cell computes the analytical posterior (Normal-InvGamma) and posterior predictive samples.
import numpy as np

def exact_bayesian_linear_regression(X_train, y_train, X_test=None, n_samples=1000, tau=10.0, a0=1e-2, b0=1e-2):
    """Posterior sampling for linear regression with conjugate Normal-Inverse-Gamma prior.

    Prior: theta | sigma2 ~ N(0, sigma2 * V0),  sigma2 ~ InvGamma(a0, b0)
    V0 = tau^2 * I provides a weak (std=tau) prior on coefficients.
    Returns posterior samples and posterior predictive mean (if X_test provided).
    """
    n, p = X_train.shape
    X_train_design = np.hstack([np.ones((n,1)), X_train])
    if X_test is not None:
        X_test_design = np.hstack([np.ones((X_test.shape[0],1)), X_test])
    else:
        X_test_design = None

    p_design = p + 1
    mu0 = np.zeros(p_design)
    V0 = (tau ** 2) * np.eye(p_design)
    V0_inv = np.linalg.inv(V0)

    XtX = X_train_design.T @ X_train_design
    Xty = X_train_design.T @ y_train

    Vn_inv = XtX + V0_inv
    Vn = np.linalg.inv(Vn_inv)
    mu_n = Vn @ Xty

    a_n = a0 + n / 2.0
    term = (y_train.T @ y_train) + (mu0.T @ V0_inv @ mu0) - (mu_n.T @ Vn_inv @ mu_n)
    b_n = b0 + 0.5 * term

    # Sample sigma2 via gamma relation and then theta conditional on sigma2
    gamma_draws = np.random.gamma(shape=a_n, scale=1.0 / b_n, size=n_samples)
    sigma2_samples = 1.0 / gamma_draws

    L = np.linalg.cholesky(Vn)
    theta_samples = np.empty((n_samples, p_design))
    for i, s2 in enumerate(sigma2_samples):
        z = np.random.normal(size=p_design)
        theta_samples[i] = mu_n + np.sqrt(s2) * (L @ z)

    if X_test_design is not None:
        preds = X_test_design @ theta_samples.T  # (n_test, n_samples)
        y_hat_mean = preds.mean(axis=1)
        return theta_samples, sigma2_samples, y_hat_mean
    return theta_samples, sigma2_samples


# Run exact inference and evaluate
theta_samps, sigma2_samps, y_hat_exact = exact_bayesian_linear_regression(X_train, y_train, X_test, n_samples=1000, tau=10.0, a0=1e-2, b0=1e-2)
print('theta_samples shape:', theta_samps.shape)
print('sigma2_samples shape:', sigma2_samps.shape)

preds_exact = y_hat_exact * y_std + y_mean
y_true = y_test * y_std + y_mean
corr, mae, rae, rmse, r2 = compute_error(y_true, preds_exact)
print('\nExact conjugate posterior predictive results:')
print('CorrCoef: %.3f\nMAE: %.3f\nRMSE: %.3f\nR2: %.3f' % (corr, mae, rmse, r2))

metrics = evaluate_model(y_true, preds_exact)
print("\n Exact Inference New Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

theta_samples shape: (1000, 18)
sigma2_samples shape: (1000,)

Exact conjugate posterior predictive results:
CorrCoef: 0.548
MAE: 13.542
RMSE: 25.553
R2: 0.296

 Exact Inference New Evaluation Metrics:
MAE: 13.5417
RMSE: 25.5534
Bias: -1.9752
sMAPE: 0.3856
R2: 0.2956
Correlation: 0.5479


## Spatial lag covariates

As a first step toward modeling correlation between neighboring cells, we can add a spatial lag term by averaging each predictor over the immediate rook neighbors of each cell. This keeps the model linear and lets us reuse the same Bayesian machinery.


In [73]:
# Build spatial lag covariates from immediate rook neighbours on the regular grid.
# This keeps the model linear: x_i is augmented with the mean of the same predictors in N(i).
# We now lag the explicit model_feature_cols list, which includes the CLR columns we registered in df.

requested_spatial_cols = list(dict.fromkeys(model_feature_cols)) if "model_feature_cols" in globals() else []
if not requested_spatial_cols:
    raise ValueError("model_feature_cols is not available yet; run the feature-construction cells first.")

spatial_base_cols = [c for c in requested_spatial_cols if c in df.columns]
missing_spatial_cols = [c for c in requested_spatial_cols if c not in df.columns]
if missing_spatial_cols:
    print("Skipping columns that are not in df and cannot be lagged directly:")
    print(missing_spatial_cols)

if not spatial_base_cols:
    raise ValueError("No spatial base columns exist in df after filtering; cannot build spatial lag features.")

if "x" not in df.columns or "y" not in df.columns:
    raise KeyError("Spatial lag features require x and y coordinate columns in df.")

# Preserve coordinates and predictors; keep coordinates numeric and rounded to avoid join noise.
spatial_df = df[["x", "y"] + spatial_base_cols].copy()
spatial_df[["x", "y"]] = spatial_df[["x", "y"]].apply(pd.to_numeric, errors="coerce")
spatial_df[spatial_base_cols] = spatial_df[spatial_base_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
spatial_df[["x", "y"]] = spatial_df[["x", "y"]].round(6)

# Infer the grid spacing from the observed coordinates.
x_vals = np.sort(spatial_df["x"].unique())
y_vals = np.sort(spatial_df["y"].unique())
x_step = np.diff(x_vals)
y_step = np.diff(y_vals)
step_candidates = np.concatenate([x_step[x_step > 0], y_step[y_step > 0]]) if (len(x_step) or len(y_step)) else np.array([])
if step_candidates.size == 0:
    raise ValueError("Could not infer a positive grid spacing from x/y coordinates.")
cell_step = float(np.min(step_candidates))

coord_df = spatial_df[["x", "y"]].copy()
lag_sum = np.zeros((len(spatial_df), len(spatial_base_cols)), dtype=float)
lag_count = np.zeros(len(spatial_df), dtype=float)
offsets = [(-cell_step, 0.0), (cell_step, 0.0), (0.0, -cell_step), (0.0, cell_step)]

for dx, dy in offsets:
    neighbor = spatial_df[["x", "y"] + spatial_base_cols].copy()
    neighbor["x"] = (neighbor["x"] - dx).round(6)
    neighbor["y"] = (neighbor["y"] - dy).round(6)

    # Make the lagged columns explicit before the merge so we do not depend on pandas suffix rules.
    neighbor = neighbor.rename(columns={c: f"{c}_nbr" for c in spatial_base_cols})
    merged = coord_df.merge(neighbor, on=["x", "y"], how="left")

    neighbor_cols = [f"{c}_nbr" for c in spatial_base_cols]
    available = merged[neighbor_cols[0]].notna().to_numpy()
    lag_count += available.astype(float)
    lag_sum += merged[neighbor_cols].fillna(0.0).to_numpy()

lag_mean = np.divide(
    lag_sum,
    lag_count[:, None],
    out=np.zeros_like(lag_sum),
    where=lag_count[:, None] > 0,
)

spatial_lag_cols = [f"{c}_lag" for c in spatial_base_cols]
X_spatial = np.hstack([spatial_df[spatial_base_cols].to_numpy(), lag_mean])
X_spatial_names = spatial_base_cols + spatial_lag_cols

print(f"Inferred grid spacing: {cell_step:g}")
print(f"Spatial lag matrix shape: {X_spatial.shape}")
print(f"Base features: {len(spatial_base_cols)}")
print(f"Lag features: {len(spatial_lag_cols)}")
print(f"Example lag feature names: {spatial_lag_cols[:5]}")


Inferred grid spacing: 12.5
Spatial lag matrix shape: (2003, 34)
Base features: 17
Lag features: 17
Example lag feature names: ['biomassa_omdrev1_lag', 'grundyta_omdrev1_lag', 'medeldiameter_omdrev1_lag', 'medelhojd_omdrev1_lag', 'p95_omdrev1_lag']


In [74]:
# Spatial lag exact Bayesian linear regression
# We reuse the conjugate exact solver above on the expanded design matrix.

X_spatial_train = X_spatial[ix_train, :]
X_spatial_test = X_spatial[ix_test, :]

# Standardize the expanded feature set separately from the base model.
X_spatial_mean = X_spatial_train.mean(axis=0)
X_spatial_std = X_spatial_train.std(axis=0) + 1e-8
X_spatial_train_std = (X_spatial_train - X_spatial_mean) / X_spatial_std
X_spatial_test_std = (X_spatial_test - X_spatial_mean) / X_spatial_std

print(f"X_spatial_train shape: {X_spatial_train_std.shape}")
print(f"X_spatial_test shape: {X_spatial_test_std.shape}")

# Exact Bayesian inference for the expanded model.
theta_spatial_samps, sigma2_spatial_samps, y_hat_spatial = exact_bayesian_linear_regression(
    X_spatial_train_std,
    y_train,
    X_spatial_test_std,
    n_samples=1000,
    tau=10.0,
    a0=1e-2,
    b0=1e-2,
)

print("theta_spatial_samps shape:", theta_spatial_samps.shape)
print("sigma2_spatial_samps shape:", sigma2_spatial_samps.shape)

preds_spatial_exact = y_hat_spatial * y_std + y_mean
y_true_spatial = y_test * y_std + y_mean
corr, mae, rae, rmse, r2 = compute_error(y_true_spatial, preds_spatial_exact)
print("\nSpatial-lag exact Bayesian results:")
print("CorrCoef: %.3f\nMAE: %.3f\nRMSE: %.3f\nR2: %.3f" % (corr, mae, rmse, r2))

metrics = evaluate_model(y_true_spatial, preds_spatial_exact, coords=df.iloc[ix_test][["x", "y"]].to_numpy())
print("\nSpatial New Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

X_spatial_train shape: (1321, 34)
X_spatial_test shape: (682, 34)
theta_spatial_samps shape: (1000, 35)
sigma2_spatial_samps shape: (1000,)

Spatial-lag exact Bayesian results:
CorrCoef: 0.532
MAE: 13.525
RMSE: 25.879
R2: 0.278

Spatial New Evaluation Metrics:
MAE: 13.5251
RMSE: 25.8790
Bias: -2.3297
sMAPE: 0.3912
R2: 0.2775
Correlation: 0.5323
Moran_I: 0.3886
Moran_p: 0.0010


c:\Users\rasmu\miniconda3\envs\mbml\Lib\site-packages\libpysal\weights\distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 4 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


## Spatial autoregressive model (SAR): $y = \rho W y + X\beta + \varepsilon$

This adds a true spatial-lag-of-target model, where neighboring response values enter directly via $Wy$.

Notes:
- This is a separate model variant and does not modify the earlier feature-lag model.
- We estimate SAR on the full filtered dataset because the spatial lag term couples observations through the graph.

In [75]:
# True SAR model: y = rho * W y + X beta + eps
# Uses PySAL maximum-likelihood spatial lag estimation.

import warnings

try:
    from spreg import ML_Lag
except ImportError as e:
    raise ImportError(
        "spreg is required for SAR modeling. Install with: pip install spreg"
    ) from e

if "model_feature_cols" not in globals() or not model_feature_cols:
    raise ValueError("model_feature_cols is not available; run feature construction cells first.")
if "x" not in df.columns or "y" not in df.columns:
    raise KeyError("SAR model requires x and y coordinate columns in df.")

# Use leakage-resistant split if available.
if "ix_train_active" in globals() and "ix_test_active" in globals():
    ix_train_sar = np.asarray(ix_train_active, dtype=int)
    ix_test_sar = np.asarray(ix_test_active, dtype=int)
    split_name = "active spatial split"
else:
    ix_train_sar = np.asarray(ix_train, dtype=int)
    ix_test_sar = np.asarray(ix_test, dtype=int)
    split_name = "original random split"

print(f"SAR using {split_name}: train={len(ix_train_sar)}, test={len(ix_test_sar)}")

# Build SAR design matrices from the same base features used above.
X_raw = (
    df[model_feature_cols]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
    .to_numpy(dtype=float)
)

# Standardize with training stats only.
X_train_raw = X_raw[ix_train_sar]
X_mean_sar = X_train_raw.mean(axis=0)
X_std_sar = X_train_raw.std(axis=0)
non_constant = X_std_sar > 1e-10

if not np.any(non_constant):
    raise ValueError("All SAR predictors are constant after preprocessing.")

X_train_scaled = (X_train_raw[:, non_constant] - X_mean_sar[non_constant]) / X_std_sar[non_constant]
X_all_scaled = (X_raw[:, non_constant] - X_mean_sar[non_constant]) / X_std_sar[non_constant]
feature_names_nonconst = [c for c, keep in zip(model_feature_cols, non_constant) if keep]

# Keep only linearly independent columns using training data.
independent_idx = []
for j in range(X_train_scaled.shape[1]):
    cand_idx = independent_idx + [j]
    if np.linalg.matrix_rank(X_train_scaled[:, cand_idx]) > len(independent_idx):
        independent_idx.append(j)

X_sar = X_all_scaled[:, independent_idx]
model_feature_cols_sar = [feature_names_nonconst[j] for j in independent_idx]

dropped_cols = [c for c in model_feature_cols if c not in model_feature_cols_sar]
if dropped_cols:
    print("Dropped unstable/collinear SAR predictors:")
    print(dropped_cols)

if X_sar.shape[1] == 0:
    raise ValueError("No valid SAR predictors left after conditioning.")

y_sar = (
    pd.to_numeric(df["medelhojd_omdrev2"], errors="coerce")
    - pd.to_numeric(df["medelhojd_omdrev1"], errors="coerce")
).fillna(0.0).to_numpy(dtype=float).reshape(-1, 1)

coords_sar = (
    df[["x", "y"]]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
    .to_numpy(dtype=float)
)

# Train-only data for fitting
X_sar_train = X_sar[ix_train_sar]
y_sar_train = y_sar[ix_train_sar]
coords_sar_train = coords_sar[ix_train_sar]
X_sar_test = X_sar[ix_test_sar]
y_sar_test = y_sar[ix_test_sar].ravel()
coords_sar_test = coords_sar[ix_test_sar]

# Build a connected KNN graph for W on TRAINING data.
n_obs = coords_sar_train.shape[0]
if n_obs < 3:
    raise ValueError("Need at least 3 training observations to fit the SAR model.")

k_candidates = [8, 12, 16, 24, 32, 48, 64, 96]
k_candidates = [k for k in k_candidates if k < n_obs]
if not k_candidates:
    k_candidates = [max(1, n_obs - 1)]

w_sar = None
last_components = None
for k_try in k_candidates:
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="The weights matrix is not fully connected.*",
            category=UserWarning,
        )
        w_try = libpysal.weights.KNN.from_array(coords_sar_train, k=k_try)
    w_try.transform = "r"
    n_components = getattr(w_try, "n_components", 1)
    last_components = n_components
    if n_components == 1:
        w_sar = w_try
        print(f"Using connected TRAIN KNN weights with k={k_try}")
        break

if w_sar is None:
    # Fallback: use the largest candidate even if still disconnected.
    k_last = k_candidates[-1]
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="The weights matrix is not fully connected.*",
            category=UserWarning,
        )
        w_sar = libpysal.weights.KNN.from_array(coords_sar_train, k=k_last)
    w_sar.transform = "r"
    print(
        f"Warning: train weights still disconnected at k={k_last} "
        f"(components={last_components}). Proceeding anyway."
    )

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="Method 'bounded' does not support relative tolerance in x.*",
        category=RuntimeWarning,
    )
    sar_model = ML_Lag(
        y_sar_train,
        X_sar_train,
        w=w_sar,
        method="ord",  # More stable/economical for many practical SAR setups.
        name_y="growth",
        name_x=model_feature_cols_sar,
    )

# In-sample train predictions from model
sar_pred_train = np.asarray(sar_model.predy).ravel()
y_sar_train_true = y_sar_train.ravel()

# Out-of-sample test predictions using reduced-form mean (I - rho W)^-1 X beta is not available without test graph coupling.
# For conservative comparison, use linear component from SAR coefficients on held-out X.
rho_hat = float(np.asarray(sar_model.rho).ravel()[0])
beta_hat = np.asarray(sar_model.betas).ravel()
intercept_hat = float(beta_hat[0])
beta_x_hat = beta_hat[1:1 + X_sar_train.shape[1]]
sar_pred_test = intercept_hat + X_sar_test @ beta_x_hat

print(f"Estimated spatial autoregressive coefficient rho: {rho_hat:.4f}")
print(f"SAR train pseudo R2 (spreg): {getattr(sar_model, 'pr2', np.nan):.4f}")

corr_tr, mae_tr, rae_tr, rmse_tr, r2_tr = compute_error(y_sar_train_true, sar_pred_train)
print("\nSAR train results:")
print("CorrCoef: %.3f\nMAE: %.3f\nRMSE: %.3f\nR2: %.3f" % (corr_tr, mae_tr, rmse_tr, r2_tr))

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="The weights matrix is not fully connected.*",
        category=UserWarning,
    )
    sar_metrics_train = evaluate_model(y_sar_train_true, sar_pred_train, w=w_sar)

print("\nSAR Train Evaluation Metrics:")
for k, v in sar_metrics_train.items():
    print(f"{k}: {v:.4f}")

corr_te, mae_te, rae_te, rmse_te, r2_te = compute_error(y_sar_test, sar_pred_test)
print("\nSAR test results (conservative linear component prediction):")
print("CorrCoef: %.3f\nMAE: %.3f\nRMSE: %.3f\nR2: %.3f" % (corr_te, mae_te, rmse_te, r2_te))

sar_metrics = evaluate_model(y_sar_test, sar_pred_test, coords=coords_sar_test)
print("\nSAR Test Evaluation Metrics:")
for k, v in sar_metrics.items():
    print(f"{k}: {v:.4f}")

# Keep compatibility with downstream cells
y_sar_true = y_sar.ravel()


SAR using active spatial split: train=1748, test=254
Dropped unstable/collinear SAR predictors:
['slu_skogskarta_tall_volym_clr']
Estimated spatial autoregressive coefficient rho: 0.3089
SAR train pseudo R2 (spreg): 0.4030

SAR train results:
CorrCoef: 0.635
MAE: 12.463
RMSE: 21.656
R2: 0.403

SAR Train Evaluation Metrics:
MAE: 12.4628
RMSE: 21.6563
Bias: -0.0000
sMAPE: 0.3732
R2: 0.4030
Correlation: 0.6348
Moran_I: 0.0442
Moran_p: 0.0010

SAR test results (conservative linear component prediction):
CorrCoef: 0.530
MAE: 16.404
RMSE: 21.144
R2: 0.000

SAR Test Evaluation Metrics:
MAE: 16.4039
RMSE: 21.1441
Bias: -8.4629
sMAPE: 0.5697
R2: -0.1655
Correlation: 0.5299
Moran_I: 0.3124
Moran_p: 0.0010


### SAR sensitivity check across fixed k values

This cell compares the SAR fit across a few fixed KNN neighborhood sizes so you can see how sensitive the result is to the choice of $W$.

In [76]:
# Compare SAR fits across a few fixed k values on TRAINING split,
# then evaluate conservative test predictions using fitted linear component.

import warnings

if "X_sar" not in globals() or "y_sar" not in globals() or "coords_sar" not in globals():
    raise ValueError("Run the SAR cell first so X_sar, y_sar, and coords_sar are available.")

if "ix_train_active" in globals() and "ix_test_active" in globals():
    ix_train_eval = np.asarray(ix_train_active, dtype=int)
    ix_test_eval = np.asarray(ix_test_active, dtype=int)
    split_name = "active spatial split"
else:
    ix_train_eval = np.asarray(ix_train, dtype=int)
    ix_test_eval = np.asarray(ix_test, dtype=int)
    split_name = "original random split"

print(f"SAR sensitivity using {split_name}: train={len(ix_train_eval)}, test={len(ix_test_eval)}")

X_train_eval = X_sar[ix_train_eval]
y_train_eval = y_sar[ix_train_eval]
coords_train_eval = coords_sar[ix_train_eval]
X_test_eval = X_sar[ix_test_eval]
y_test_eval = y_sar[ix_test_eval].ravel()
coords_test_eval = coords_sar[ix_test_eval]

k_values = [8, 16, 32, 64]
rows = []

for k in k_values:
    if k >= len(coords_train_eval):
        continue

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="The weights matrix is not fully connected.*",
            category=UserWarning,
        )
        w_k = libpysal.weights.KNN.from_array(coords_train_eval, k=k)
    w_k.transform = "r"

    n_components = getattr(w_k, "n_components", 1)

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Method 'bounded' does not support relative tolerance in x.*",
            category=RuntimeWarning,
        )
        sar_k = ML_Lag(
            y_train_eval,
            X_train_eval,
            w=w_k,
            method="ord",
            name_y="growth",
            name_x=model_feature_cols_sar,
        )

    pred_train_k = np.asarray(sar_k.predy).ravel()
    true_train_k = y_train_eval.ravel()

    # Conservative test prediction from linear component only.
    beta_hat = np.asarray(sar_k.betas).ravel()
    intercept_hat = float(beta_hat[0])
    beta_x_hat = beta_hat[1:1 + X_train_eval.shape[1]]
    pred_test_k = intercept_hat + X_test_eval @ beta_x_hat

    corr_tr, mae_tr, rae_tr, rmse_tr, r2_tr = compute_error(true_train_k, pred_train_k)
    corr_te, mae_te, rae_te, rmse_te, _ = compute_error(y_test_eval, pred_test_k)
    test_metrics = evaluate_model(y_test_eval, pred_test_k, coords=coords_test_eval)

    rows.append({
        "k": k,
        "components_train": int(n_components),
        "rho": float(np.asarray(sar_k.rho).ravel()[0]),
        "pseudo_R2_train": float(getattr(sar_k, "pr2", np.nan)),
        "R2_train": float(r2_tr),
        "R2_test_raw": float(test_metrics["R2"]),
        "RMSE_test": float(rmse_te),
        "MAE_test": float(mae_te),
        "Corr_test": float(corr_te),
        **{f"test_{kk}": vv for kk, vv in test_metrics.items()},
    })

sar_k_comparison = pd.DataFrame(rows)
print(sar_k_comparison.to_string(index=False))

best_row = sar_k_comparison.sort_values(["R2_test_raw", "Corr_test"], ascending=False).iloc[0]
print(
    f"\nBest on held-out test: k={int(best_row['k'])}, "
    f"rho={best_row['rho']:.4f}, R2_test_raw={best_row['R2_test_raw']:.4f}, "
    f"test_Moran_I={best_row['test_Moran_I']:.4f}"
)


SAR sensitivity using active spatial split: train=1748, test=254
 k  components_train      rho  pseudo_R2_train  R2_train  R2_test_raw  RMSE_test  MAE_test  Corr_test  test_MAE  test_RMSE  test_Bias  test_sMAPE   test_R2  test_Correlation  test_Moran_I  test_Moran_p
 8                 5 0.542102         0.532409  0.531293    -0.575886  24.586335 18.898341   0.544530 18.898341  24.586335 -17.974609    0.737885 -0.575886          0.544530      0.141498         0.001
16                 3 0.539838         0.478176  0.477758    -0.552608  24.404071 18.801887   0.539265 18.801887  24.404071 -17.364144    0.751377 -0.552608          0.539265      0.173290         0.001
32                 3 0.391286         0.424858  0.424829    -0.233230  21.749710 16.709645   0.533767 16.709645  21.749710 -11.956312    0.608829 -0.233230          0.533767      0.244523         0.001
64                 2 0.326921         0.406754  0.406750    -0.166198  21.150352 16.379576   0.530684 16.379576  21.150352  -9.

## Hierarchical spatial model with inferred groups

This cell implements a hierarchical model where group intercepts are inferred from spatial proximity clustering:

$$y_i = \alpha_{g(i)} + X_i^T\beta + \bar{X}_{N(i)}^T\theta + \varepsilon_i$$

Groups $g(i)$ are determined by k-means clustering on spatial coordinates. Neighbor covariate means $\bar{X}_{N(i)}$ are computed using the same weight matrix $W$ from the SAR model. This allows us to estimate both direct covariate effects ($\beta$) and spatial neighbor effects ($\theta$) while accounting for spatial heterogeneity through group intercepts.


In [77]:
from sklearn.cluster import KMeans
import warnings

# Hierarchical model with spatial group intercepts and neighbor covariate effects
# Model form: y_i = alpha_{g(i)} + X_i^T * beta + X_bar_{N(i)}^T * theta + epsilon_i
# Groups inferred from spatial proximity via k-means clustering on coordinates
# Uses active split when available to provide train/test separated evaluation.

if "X_sar" not in globals() or "y_sar_true" not in globals() or "coords_sar" not in globals() or "w_sar" not in globals():
    raise ValueError("Run the SAR cell first so X_sar, y_sar_true, coords_sar, and w_sar are available.")

if "ix_train_active" in globals() and "ix_test_active" in globals():
    ix_train_h = np.asarray(ix_train_active, dtype=int)
    ix_test_h = np.asarray(ix_test_active, dtype=int)
    split_name = "active spatial split"
else:
    ix_train_h = np.asarray(ix_train, dtype=int)
    ix_test_h = np.asarray(ix_test, dtype=int)
    split_name = "original random split"

print(f"Hierarchical model using {split_name}: train={len(ix_train_h)}, test={len(ix_test_h)}")

X_train_h = X_sar[ix_train_h]
X_test_h = X_sar[ix_test_h]
y_train_h = y_sar_true[ix_train_h]
y_test_h = y_sar_true[ix_test_h]
coords_train_h = coords_sar[ix_train_h]
coords_test_h = coords_sar[ix_test_h]

# --- Step 1: Infer spatial groups via k-means clustering on training coordinates ---
n_groups = max(3, int(np.sqrt(len(y_train_h))))  # Default: sqrt(n), at least 3
print(f"Inferring {n_groups} spatial groups from TRAIN coordinates using k-means...")
kmeans = KMeans(n_clusters=n_groups, random_state=42, n_init=10)
group_labels_train = kmeans.fit_predict(coords_train_h)
group_labels_test = kmeans.predict(coords_test_h)
print(f"Group assignment complete (train counts): {np.bincount(group_labels_train)}")

# --- Step 2: Compute spatial lag of covariates X_bar_{N(i)} using train neighbors ---
print("\nComputing spatial neighbor covariate means...")

train_pos = {idx: pos for pos, idx in enumerate(ix_train_h)}

X_neighbor_mean_train = np.zeros_like(X_train_h)
for p, global_idx in enumerate(ix_train_h):
    neighbors = w_sar.neighbors.get(int(global_idx), [])
    neighbors_train = [n for n in neighbors if n in train_pos]
    if neighbors_train:
        locs = [train_pos[n] for n in neighbors_train]
        X_neighbor_mean_train[p, :] = X_train_h[locs, :].mean(axis=0)

X_neighbor_mean_test = np.zeros_like(X_test_h)
for p, global_idx in enumerate(ix_test_h):
    neighbors = w_sar.neighbors.get(int(global_idx), [])
    neighbors_train = [n for n in neighbors if n in train_pos]
    if neighbors_train:
        locs = [train_pos[n] for n in neighbors_train]
        X_neighbor_mean_test[p, :] = X_train_h[locs, :].mean(axis=0)

print(f"X_neighbor_mean_train shape: {X_neighbor_mean_train.shape}")
print(f"X_neighbor_mean_test shape: {X_neighbor_mean_test.shape}")

# --- Step 3: Construct design matrices ---
# Create group dummy variables (drop first to avoid multicollinearity)
group_dummies_train = pd.get_dummies(group_labels_train, prefix="group", drop_first=True)
group_dummies_test = pd.get_dummies(group_labels_test, prefix="group", drop_first=True)

# Align test dummy columns to train columns for stable matrix shape.
group_dummies_test = group_dummies_test.reindex(columns=group_dummies_train.columns, fill_value=0)

group_dummies_train = group_dummies_train.to_numpy(dtype=float)
group_dummies_test = group_dummies_test.to_numpy(dtype=float)

X_hierarchical_train = np.hstack([group_dummies_train, X_train_h, X_neighbor_mean_train])
X_hierarchical_test = np.hstack([group_dummies_test, X_test_h, X_neighbor_mean_test])

print(f"X_hierarchical_train shape: {X_hierarchical_train.shape}")
print(f"X_hierarchical_test shape: {X_hierarchical_test.shape}")

# --- Step 4: Fit hierarchical model via OLS on training data ---
print("\nFitting hierarchical model on training data...")
X_hierarchical_design_train = np.hstack([np.ones((len(y_train_h), 1)), X_hierarchical_train])
XtX = X_hierarchical_design_train.T @ X_hierarchical_design_train
Xty = X_hierarchical_design_train.T @ y_train_h

try:
    beta_hier = np.linalg.solve(XtX, Xty)
except np.linalg.LinAlgError:
    print("Warning: XtX is singular, using least-squares solution.")
    beta_hier = np.linalg.lstsq(XtX, Xty, rcond=None)[0]

y_pred_hier_train = X_hierarchical_design_train @ beta_hier
X_hierarchical_design_test = np.hstack([np.ones((len(y_test_h), 1)), X_hierarchical_test])
y_pred_hier_test = X_hierarchical_design_test @ beta_hier

# --- Step 5: Evaluate hierarchical model (train + test) ---
print("\nHierarchical model train results:")
corr_tr, mae_tr, rae_tr, rmse_tr, r2_tr = compute_error(y_train_h, y_pred_hier_train)
print(f"CorrCoef: {corr_tr:.4f}\nMAE: {mae_tr:.4f}\nRMSE: {rmse_tr:.4f}\nR2: {r2_tr:.4f}")

print("\nHierarchical model test results:")
corr_te, mae_te, rae_te, rmse_te, r2_te = compute_error(y_test_h, y_pred_hier_test)
print(f"CorrCoef: {corr_te:.4f}\nMAE: {mae_te:.4f}\nRMSE: {rmse_te:.4f}\nR2: {r2_te:.4f}")

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="The weights matrix is not fully connected.*",
        category=UserWarning,
    )
    hier_metrics = evaluate_model(y_test_h, y_pred_hier_test, coords=coords_test_h)

print("\nHierarchical Model Test Evaluation Metrics:")
for k, v in hier_metrics.items():
    print(f"{k}: {v:.4f}")

# Keep compatibility with downstream GP cell naming expectations
y_pred_hier = y_pred_hier_train


Hierarchical model using active spatial split: train=1748, test=254
Inferring 41 spatial groups from TRAIN coordinates using k-means...
Group assignment complete (train counts): [42 76 73 34 30 46 42 38 55 49 30 42 57 28 54 70 15 41 31 38 40 36 48 45
 56 68 71 49 61 39 31 32 33 21 37 28 22 55 28 46 11]

Computing spatial neighbor covariate means...
X_neighbor_mean_train shape: (1748, 16)
X_neighbor_mean_test shape: (254, 16)
X_hierarchical_train shape: (1748, 72)
X_hierarchical_test shape: (254, 72)

Fitting hierarchical model on training data...

Hierarchical model train results:
CorrCoef: 0.6902
MAE: 11.9907
RMSE: 20.2799
R2: 0.4764

Hierarchical model test results:
CorrCoef: 0.5220
MAE: 17.3193
RMSE: 22.1428
R2: 0.0000

Hierarchical Model Test Evaluation Metrics:
MAE: 17.3193
RMSE: 22.1428
Bias: -5.1462
sMAPE: 0.5787
R2: -0.2782
Correlation: 0.5220
Moran_I: 0.3981
Moran_p: 0.0010


c:\Users\rasmu\miniconda3\envs\mbml\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=7.
  warnings.warn(


## GP-augmented hierarchical spatial model with Matérn 3/2 kernel

This cell extends the hierarchical model with a spatial Gaussian process to capture fine-grained spatial structure:

$$y_i = \alpha_{g(i)} + X_i^T\beta + \bar{X}_{N(i)}^T\theta + f(\mathbf{s}_i) + \varepsilon_i$$

The spatial effect $f(\mathbf{s}_i)$ is a GP with Matérn 3/2 kernel: $k(d) = \sigma_f^2\left(1 + \frac{\sqrt{3}d}{\ell}\right)\exp\left(-\frac{\sqrt{3}d}{\ell}\right)$.

The GP is fit to the residuals of the hierarchical model, allowing us to capture spatial patterns not explained by group intercepts and covariate effects.


In [78]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C
import warnings

# GP-augmented hierarchical model with Matérn 3/2 kernel
# Model: y_i = alpha_{g(i)} + X_i^T * beta + X_bar_{N(i)}^T * theta + f(s_i) + epsilon_i
# PROPER OUT-OF-SAMPLE EVALUATION: train on train indices, evaluate on test indices

if "y_sar_true" not in globals() or "X_sar" not in globals() or "coords_sar" not in globals() or "w_sar" not in globals():
    raise ValueError("Run the SAR cell first so y_sar_true, X_sar, coords_sar, and w_sar are available.")

if "ix_train" not in globals() or "ix_test" not in globals():
    raise ValueError("Train/test split (ix_train, ix_test) not found. Run the data splitting cell first.")

print("="*70)
print("GP-AUGMENTED HIERARCHICAL MODEL (Matérn 3/2 kernel)")
print("PROPER OUT-OF-SAMPLE EVALUATION")
print("="*70)

# --- Step 0: Split all data by train/test indices ---
print(f"\nUsing existing train/test split:")
print(f"  Train: {len(ix_train)} samples")
print(f"  Test: {len(ix_test)} samples")

X_sar_train = X_sar[ix_train, :]
X_sar_test = X_sar[ix_test, :]
y_sar_train = y_sar_true[ix_train]
y_sar_test = y_sar_true[ix_test]
coords_sar_train = coords_sar[ix_train, :]
coords_sar_test = coords_sar[ix_test, :]

# --- Step 1: Rebuild hierarchical model on TRAINING data only ---
print("\n--- Training Hierarchical Model (on train data) ---")
from sklearn.cluster import KMeans

n_groups = max(3, int(np.sqrt(len(y_sar_train))))
kmeans = KMeans(n_clusters=n_groups, random_state=42, n_init=10)
group_labels_train = kmeans.fit_predict(coords_sar_train)
group_labels_test = kmeans.predict(coords_sar_test)  # Predict groups for test data
print(f"Inferred {n_groups} spatial groups; train distribution: {np.bincount(group_labels_train)[:5]}...")

# Compute neighbor means on training data
n_obs_train, n_features = X_sar_train.shape
X_neighbor_mean_train = np.zeros_like(X_sar_train)
for i in range(n_obs_train):
    neighbors = w_sar.neighbors.get(ix_train[i], [])
    # Filter neighbors to only those in training set
    neighbors_in_train = [n for n in neighbors if n in ix_train]
    if len(neighbors_in_train) > 0:
        neighbor_indices = [np.where(ix_train == n)[0][0] for n in neighbors_in_train]
        X_neighbor_mean_train[i, :] = X_sar_train[neighbor_indices, :].mean(axis=0)

# Build and fit hierarchical model on training data
group_dummies_train = pd.get_dummies(group_labels_train, prefix="group", drop_first=True).to_numpy(dtype=float)
X_hierarchical_train = np.hstack([group_dummies_train, X_sar_train, X_neighbor_mean_train])
X_hierarchical_design_train = np.hstack([np.ones((len(y_sar_train), 1)), X_hierarchical_train])

XtX_train = X_hierarchical_design_train.T @ X_hierarchical_design_train
Xty_train = X_hierarchical_design_train.T @ y_sar_train

try:
    beta_hier_train = np.linalg.solve(XtX_train, Xty_train)
except np.linalg.LinAlgError:
    print("Warning: XtX is singular, using least-squares solution.")
    beta_hier_train = np.linalg.lstsq(XtX_train, Xty_train, rcond=None)[0]

# Make predictions on test data
group_dummies_test = pd.get_dummies(group_labels_test, prefix="group", drop_first=True).to_numpy(dtype=float)
X_neighbor_mean_test = np.zeros_like(X_sar_test)
for i in range(len(ix_test)):
    neighbors = w_sar.neighbors.get(ix_test[i], [])
    neighbors_in_train = [n for n in neighbors if n in ix_train]
    if len(neighbors_in_train) > 0:
        neighbor_indices = [np.where(ix_train == n)[0][0] for n in neighbors_in_train]
        X_neighbor_mean_test[i, :] = X_sar_train[neighbor_indices, :].mean(axis=0)

X_hierarchical_test = np.hstack([group_dummies_test, X_sar_test, X_neighbor_mean_test])
X_hierarchical_design_test = np.hstack([np.ones((len(y_sar_test), 1)), X_hierarchical_test])
y_pred_hier_test = X_hierarchical_design_test @ beta_hier_train

print(f"Hierarchical model train R2: {r2_score(y_sar_train, X_hierarchical_design_train @ beta_hier_train):.4f}")
print(f"Hierarchical model test R2: {r2_score(y_sar_test, y_pred_hier_test):.4f}")

# --- Step 2: Compute residuals on TRAINING data and fit GP ---
print("\n--- Fitting GP (Matérn 3/2) on training residuals ---")
residuals_hier_train = y_sar_train - (X_hierarchical_design_train @ beta_hier_train)
print(f"Hierarchical residuals (train): Mean={residuals_hier_train.mean():.4f}, Std={residuals_hier_train.std():.4f}")

kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e2), nu=1.5)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,
    optimizer='fmin_l_bfgs_b',
    n_restarts_optimizer=5,
    normalize_y=False,
)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    gp.fit(coords_sar_train, residuals_hier_train)

print(f"Fitted kernel: {gp.kernel_}")
print(f"Log-marginal-likelihood (train): {gp.log_marginal_likelihood(gp.kernel_.theta):.4f}")

# --- Step 3: GP predictions on TEST data ---
print("\n--- Evaluating on test data ---")
gp_pred_train, gp_std_train = gp.predict(coords_sar_train, return_std=True)
gp_pred_test, gp_std_test = gp.predict(coords_sar_test, return_std=True)

print(f"GP train predictions: Mean={gp_pred_train.mean():.4f}, Std={gp_pred_train.std():.4f}")
print(f"GP test predictions: Mean={gp_pred_test.mean():.4f}, Std={gp_pred_test.std():.4f}")
print(f"GP test uncertainty: Mean={gp_std_test.mean():.4f}, Range=[{gp_std_test.min():.4f}, {gp_std_test.max():.4f}]")

# --- Step 4: Combined predictions on TEST data ---
y_pred_gp_hier_test = y_pred_hier_test + gp_pred_test

# --- Step 5: Evaluate all models on TEST set ---
print("\n" + "-"*70)
print("EVALUATION ON TEST SET")
print("-"*70)

# Hierarchical only
print("\nHierarchical model (test):")
corr_h, mae_h, rae_h, rmse_h, r2_h = compute_error(y_sar_test, y_pred_hier_test)
print(f"CorrCoef: {corr_h:.4f}, MAE: {mae_h:.4f}, RMSE: {rmse_h:.4f}, R2: {r2_h:.4f}")

# GP-Hierarchical
print("\nGP-Hierarchical model (test):")
corr_gp, mae_gp, rae_gp, rmse_gp, r2_gp = compute_error(y_sar_test, y_pred_gp_hier_test)
print(f"CorrCoef: {corr_gp:.4f}, MAE: {mae_gp:.4f}, RMSE: {rmse_gp:.4f}, R2: {r2_gp:.4f}")

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="The weights matrix is not fully connected.*")
    gp_hier_metrics_test = evaluate_model(y_sar_test, y_pred_gp_hier_test, coords=coords_sar_test)

print("\nGP-Hierarchical Model (test) Full Metrics:")
for k, v in gp_hier_metrics_test.items():
    print(f"  {k}: {v:.4f}")

# --- Step 6: Comprehensive comparison ---
print("\n" + "="*70)
print("MODEL COMPARISON ON TEST SET")
print("="*70)
print(f"{'Metric':<20} {'Hierarchical':<18} {'GP-Hierarchical':<18} {'Improvement':<12}")
print("-"*70)
print(f"{'R2':<20} {r2_h:<18.4f} {r2_gp:<18.4f} {r2_gp - r2_h:+.4f}")
print(f"{'RMSE':<20} {rmse_h:<18.4f} {rmse_gp:<18.4f} {rmse_gp - rmse_h:+.4f}")
print(f"{'MAE':<20} {mae_h:<18.4f} {mae_gp:<18.4f} {mae_gp - mae_h:+.4f}")
print(f"{'Correlation':<20} {corr_h:<18.4f} {corr_gp:<18.4f} {corr_gp - corr_h:+.4f}")
print(f"{'Moran_I':<20} {gp_hier_metrics_test['Moran_I']:<18.4f} (spatial autocorr)")
print("="*70)

print("\nNote: GP adds spatial structure f(s_i) fit from training residuals,")
print("      providing realistic uncertainty estimates on unseen test locations.")


GP-AUGMENTED HIERARCHICAL MODEL (Matérn 3/2 kernel)
PROPER OUT-OF-SAMPLE EVALUATION

Using existing train/test split:
  Train: 1321 samples
  Test: 682 samples

--- Training Hierarchical Model (on train data) ---
Inferred 36 spatial groups; train distribution: [33 26 46 35 29]...


c:\Users\rasmu\miniconda3\envs\mbml\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(


Hierarchical model train R2: 0.4823
Hierarchical model test R2: 0.3492

--- Fitting GP (Matérn 3/2) on training residuals ---
Hierarchical residuals (train): Mean=-0.0000, Std=18.1488
Fitted kernel: 18.4**2 * Matern(length_scale=9.04, nu=1.5)
Log-marginal-likelihood (train): -5666.2082

--- Evaluating on test data ---
GP train predictions: Mean=0.0000, Std=18.1488
GP test predictions: Mean=-0.0580, Std=5.6530
GP test uncertainty: Mean=17.0574, Range=[15.4918, 18.3946]

----------------------------------------------------------------------
EVALUATION ON TEST SET
----------------------------------------------------------------------

Hierarchical model (test):
CorrCoef: 0.5926, MAE: 13.0602, RMSE: 24.5609, R2: 0.3492

GP-Hierarchical model (test):
CorrCoef: 0.6357, MAE: 12.1891, RMSE: 23.5592, R2: 0.4012

GP-Hierarchical Model (test) Full Metrics:
  MAE: 12.1891
  RMSE: 23.5592
  Bias: -1.3833
  sMAPE: 0.3625
  R2: 0.4012
  Correlation: 0.6357
  Moran_I: 0.3125
  Moran_p: 0.0010

MODEL C

In [81]:
# Latent-variable model: unsupervised z <- W_i (environmental features), then y = g(z)
# Uses PCA for latent encoding (train-only), then Ridge regression from z->y.
# Evaluated using the active spatial split (ix_train_active/ix_test_active) for spatial CV.

from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
import warnings

# Choose environmental features W_i: use X_sar (standardized earlier)
if "X_sar" not in globals():
    raise ValueError("X_sar not found; run SAR preparation cell first.")

# Use active split if available
if "ix_train_active" in globals() and "ix_test_active" in globals():
    ix_tr = np.asarray(ix_train_active, dtype=int)
    ix_te = np.asarray(ix_test_active, dtype=int)
    split_label = "active spatial split"
else:
    ix_tr = np.asarray(ix_train, dtype=int)
    ix_te = np.asarray(ix_test, dtype=int)
    split_label = "original random split"

print(f"Latent model using {split_label}: train={len(ix_tr)}, test={len(ix_te)}")

W_all = X_sar.copy()  # environmental features already standardized w.r.t. train stats earlier
W_tr = W_all[ix_tr]
W_te = W_all[ix_te]

y_all = y_sar_true
y_tr = y_all[ix_tr]
y_te = y_all[ix_te]

# Determine number of latent dims
n_features = W_tr.shape[1]
n_samples_tr = W_tr.shape[0]
max_latent = min(10, n_features, int(np.sqrt(n_samples_tr)))
latent_dim = max(2, max_latent)

print(f"Training PCA latent encoder with {latent_dim} components (train samples={n_samples_tr})")
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    pca = PCA(n_components=latent_dim, random_state=42)
    z_tr = pca.fit_transform(W_tr)
    z_te = pca.transform(W_te)

explained = pca.explained_variance_ratio_.cumsum()
print(f"Explained variance by components (cumulative): {explained}")

# Fit supervised mapping g: z -> y using Ridge on training set
reg = Ridge(alpha=1.0)
reg.fit(z_tr, y_tr.ravel())

# Predictions
y_pred_tr = reg.predict(z_tr)
y_pred_te = reg.predict(z_te)

# Evaluate
corr_tr, mae_tr, rae_tr, rmse_tr, r2_tr = compute_error(y_tr.ravel(), y_pred_tr)
print("\nLatent model TRAIN results:")
print(f"CorrCoef: {corr_tr:.4f}\nMAE: {mae_tr:.4f}\nRMSE: {rmse_tr:.4f}\nR2: {r2_tr:.4f}")

corr_te, mae_te, rae_te, rmse_te, r2_te = compute_error(y_te.ravel(), y_pred_te)
print("\nLatent model TEST results:")
print(f"CorrCoef: {corr_te:.4f}\nMAE: {mae_te:.4f}\nRMSE: {rmse_te:.4f}\nR2: {r2_te:.4f}")

# Moran on residuals at test locations (uses coords)
coords = coords_sar[ix_te]
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="The weights matrix is not fully connected.*", category=UserWarning)
    latent_metrics_test = evaluate_model(y_te.ravel(), y_pred_te, coords=coords)

print("\nLatent model TEST evaluation metrics (including spatial stats):")
for k, v in latent_metrics_test.items():
    print(f"{k}: {v:.4f}")

# Save latent encoder and regressor for downstream use
pca_latent = pca
reg_latent = reg
z_train = z_tr
z_test = z_te
print("\nSaved variables: pca_latent, reg_latent, z_train, z_test")

Latent model using active spatial split: train=1748, test=254
Training PCA latent encoder with 10 components (train samples=1748)
Explained variance by components (cumulative): [0.42637179 0.59483129 0.68319931 0.75560184 0.81684827 0.87262917
 0.91109061 0.93879696 0.96288107 0.97588946]

Latent model TRAIN results:
CorrCoef: 0.5910
MAE: 13.2798
RMSE: 22.6088
R2: 0.3493

Latent model TEST results:
CorrCoef: 0.5060
MAE: 15.0529
RMSE: 19.0899
R2: 0.0500

Latent model TEST evaluation metrics (including spatial stats):
MAE: 15.0529
RMSE: 19.0899
Bias: -0.1229
sMAPE: 0.4341
R2: 0.0500
Correlation: 0.5060
Moran_I: 0.3443
Moran_p: 0.0010

Saved variables: pca_latent, reg_latent, z_train, z_test


## Probabilistic PCA latent model

This cell replaces the deterministic latent encoding with a probabilistic PCA (PPCA) latent variable model:

$$x_i = \mu + W z_i + \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, \sigma^2 I)$$

The latent state $z_i$ is inferred from the environmental parameters on the training data and then used to predict growth with a supervised model $g(z_i)$.


In [83]:
import warnings
from sklearn.linear_model import Ridge

# Probabilistic PCA (PPCA) latent model
# We fit PPCA on the training environmental features, infer latent z for train/test,
# then fit a supervised growth model y = g(z) and evaluate on the active spatial split.

if "X_sar" not in globals() or "y_sar_true" not in globals():
    raise ValueError("X_sar and y_sar_true are required; run the SAR preparation cell first.")

# Prefer the leakage-resistant active split when available.
if "ix_train_active" in globals() and "ix_test_active" in globals():
    ix_ppca_train = np.asarray(ix_train_active, dtype=int)
    ix_ppca_test = np.asarray(ix_test_active, dtype=int)
    split_label = "active spatial split"
else:
    ix_ppca_train = np.asarray(ix_train, dtype=int)
    ix_ppca_test = np.asarray(ix_test, dtype=int)
    split_label = "original random split"

print(f"PPCA latent model using {split_label}: train={len(ix_ppca_train)}, test={len(ix_ppca_test)}")

# Environmental inputs W_i: use the standardized SAR feature matrix.
W_all = X_sar
W_train = W_all[ix_ppca_train]
W_test = W_all[ix_ppca_test]
y_train = y_sar_true[ix_ppca_train]
y_test = y_sar_true[ix_ppca_test]

# Choose latent dimension q by cumulative explained variance on the training data.
def choose_latent_dim_by_variance(X, threshold=0.95, min_dim=1, max_dim=None):
    Xc = X - X.mean(axis=0)
    _, S, _ = np.linalg.svd(Xc, full_matrices=False)
    eigenvalues = (S**2) / X.shape[0]
    total = float(np.sum(eigenvalues))
    if total <= 0:
        return min_dim, np.array([1.0])
    cumulative = np.cumsum(eigenvalues) / total
    q = int(np.searchsorted(cumulative, threshold) + 1)
    q = max(min_dim, q)
    if max_dim is not None:
        q = min(q, max_dim)
    return q, cumulative

variance_threshold = 0.95
latent_dim, cumulative_var = choose_latent_dim_by_variance(
    W_train,
    threshold=variance_threshold,
    min_dim=2,
    max_dim=min(10, W_train.shape[1]),
)
print(f"Chosen latent dimension q={latent_dim} using cumulative explained variance threshold={variance_threshold:.2f}")
print(f"Cumulative explained variance (first {min(len(cumulative_var), latent_dim)} components): {cumulative_var[:latent_dim]}")

# PPCA closed-form solution (Tipping & Bishop):
# 1) Estimate mu and centered covariance on train
mu_ppca = W_train.mean(axis=0)
Xc = W_train - mu_ppca

# 2) SVD of centered training data
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
# Eigenvalues of sample covariance
eigenvalues = (S**2) / W_train.shape[0]
Uq = Vt[:latent_dim].T
lambda_q = eigenvalues[:latent_dim]

# 3) Estimate isotropic noise variance sigma^2 from remaining eigenvalues
if latent_dim < len(eigenvalues):
    sigma2_ppca = float(np.mean(eigenvalues[latent_dim:]))
else:
    sigma2_ppca = 1e-6
sigma2_ppca = max(sigma2_ppca, 1e-8)

# 4) Estimate loading matrix W_ppca
# W = U_q (Lambda_q - sigma^2 I)^{1/2} R, choose R=I for a canonical solution.
scales = np.sqrt(np.maximum(lambda_q - sigma2_ppca, 1e-8))
W_ppca = Uq * scales[np.newaxis, :]

# 5) Infer posterior mean of latent z for each observation
# E[z|x] = M^{-1} W^T (x - mu), M = W^T W + sigma^2 I
M = W_ppca.T @ W_ppca + sigma2_ppca * np.eye(latent_dim)
M_inv = np.linalg.inv(M)

z_train = (Xc @ W_ppca) @ M_inv
z_test = ((W_test - mu_ppca) @ W_ppca) @ M_inv

# 6) Supervised mapping g(z) -> y using Ridge regression
reg_ppca = Ridge(alpha=1.0)
reg_ppca.fit(z_train, y_train)

y_pred_train = reg_ppca.predict(z_train)
y_pred_test = reg_ppca.predict(z_test)

# Evaluation
corr_tr, mae_tr, rae_tr, rmse_tr, r2_tr = compute_error(y_train, y_pred_train)
corr_te, mae_te, rae_te, rmse_te, r2_te = compute_error(y_test, y_pred_test)

print("\nPPCA latent model TRAIN results:")
print(f"CorrCoef: {corr_tr:.4f}\nMAE: {mae_tr:.4f}\nRMSE: {rmse_tr:.4f}\nR2: {r2_tr:.4f}")

print("\nPPCA latent model TEST results:")
print(f"CorrCoef: {corr_te:.4f}\nMAE: {mae_te:.4f}\nRMSE: {rmse_te:.4f}\nR2: {r2_te:.4f}")

coords_test = coords_sar[ix_ppca_test]
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="The weights matrix is not fully connected.*", category=UserWarning)
    ppca_metrics_test = evaluate_model(y_test, y_pred_test, coords=coords_test)

print("\nPPCA latent model TEST evaluation metrics:")
for k, v in ppca_metrics_test.items():
    print(f"{k}: {v:.4f}")

print("\nPPCA details:")
print(f"Estimated sigma^2: {sigma2_ppca:.6f}")
print(f"Loading matrix shape: {W_ppca.shape}")
print(f"Latent z shapes: train={z_train.shape}, test={z_test.shape}")

# Save variables for downstream experiments
mu_ppca_latent = mu_ppca
W_ppca_latent = W_ppca
M_ppca_inv = M_inv
reg_ppca_latent = reg_ppca
z_train_ppca = z_train
z_test_ppca = z_test


PPCA latent model using active spatial split: train=1748, test=254
Chosen latent dimension q=9 using cumulative explained variance threshold=0.95
Cumulative explained variance (first 9 components): [0.42637179 0.59483129 0.68319931 0.75560184 0.81684827 0.87262917
 0.91109061 0.93879696 0.96288107]

PPCA latent model TRAIN results:
CorrCoef: 0.5811
MAE: 13.4962
RMSE: 22.8097
R2: 0.3377

PPCA latent model TEST results:
CorrCoef: 0.5014
MAE: 14.6020
RMSE: 18.7604
R2: 0.0825

PPCA latent model TEST evaluation metrics:
MAE: 14.6020
RMSE: 18.7604
Bias: -0.3561
sMAPE: 0.4249
R2: 0.0825
Correlation: 0.5014
Moran_I: 0.3096
Moran_p: 0.0010

PPCA details:
Estimated sigma^2: 0.084843
Loading matrix shape: (16, 9)
Latent z shapes: train=(1748, 9), test=(254, 9)


## Gaussian process on the latent state

This cell keeps the PPCA latent representation but replaces the supervised map $g(z)$ with a Gaussian process:

$$y_i = g(z_i) + \varepsilon_i$$

Here, $z_i$ is the latent state inferred from the environmental variables, and the GP captures nonlinear variation in growth across latent site-quality dimensions.


In [84]:
import warnings
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, WhiteKernel

# Gaussian process on latent state
# Reuse PPCA latent coordinates and fit a nonlinear GP map g(z) -> y.
# This is a compact alternative to the linear Ridge head used in the PPCA cell.

if "z_train_ppca" not in globals() or "z_test_ppca" not in globals():
    raise ValueError("Run the PPCA latent model cell first so z_train_ppca and z_test_ppca are available.")
if "y_sar_true" not in globals():
    raise ValueError("y_sar_true is required; run the SAR preparation cell first.")

# Use the same active split as the other spatially aware latent models.
if "ix_train_active" in globals() and "ix_test_active" in globals():
    ix_gp_train = np.asarray(ix_train_active, dtype=int)
    ix_gp_test = np.asarray(ix_test_active, dtype=int)
    split_label = "active spatial split"
else:
    ix_gp_train = np.asarray(ix_train, dtype=int)
    ix_gp_test = np.asarray(ix_test, dtype=int)
    split_label = "original random split"

print(f"GP-on-latent model using {split_label}: train={len(ix_gp_train)}, test={len(ix_gp_test)}")

# Align latent states and targets to the same active split.
z_train = z_train_ppca
z_test = z_test_ppca
y_train = y_sar_true[ix_gp_train]
y_test = y_sar_true[ix_gp_test]

# Fit a GP on latent space. Matérn 3/2 gives a smooth but not overly rigid nonlinear map.
# WhiteKernel absorbs small observation noise on top of the latent representation.
latent_dim = z_train.shape[1]
length_scale_init = np.ones(latent_dim)
kernel = (
    C(1.0, (1e-3, 1e3))
    * Matern(length_scale=length_scale_init, length_scale_bounds=(1e-2, 1e2), nu=1.5)
    + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-5, 1e2))
)

gp_latent = GaussianProcessRegressor(
    kernel=kernel,
    alpha=0.0,
    normalize_y=True,
    n_restarts_optimizer=3,
    random_state=42,
)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    gp_latent.fit(z_train, y_train)

print(f"Fitted latent-space kernel: {gp_latent.kernel_}")

# Predictions and uncertainty.
y_pred_train, y_std_train = gp_latent.predict(z_train, return_std=True)
y_pred_test, y_std_test = gp_latent.predict(z_test, return_std=True)

# Evaluate train/test.
corr_tr, mae_tr, rae_tr, rmse_tr, r2_tr = compute_error(y_train, y_pred_train)
corr_te, mae_te, rae_te, rmse_te, r2_te = compute_error(y_test, y_pred_test)

print("\nGP-latent model TRAIN results:")
print(f"CorrCoef: {corr_tr:.4f}\nMAE: {mae_tr:.4f}\nRMSE: {rmse_tr:.4f}\nR2: {r2_tr:.4f}")
print("\nGP-latent model TEST results:")
print(f"CorrCoef: {corr_te:.4f}\nMAE: {mae_te:.4f}\nRMSE: {rmse_te:.4f}\nR2: {r2_te:.4f}")
print(f"\nAverage predictive std on test: {y_std_test.mean():.4f}")

coords_test = coords_sar[ix_gp_test]
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="The weights matrix is not fully connected.*", category=UserWarning)
    gp_latent_metrics_test = evaluate_model(y_test, y_pred_test, coords=coords_test)

print("\nGP-latent model TEST evaluation metrics:")
for k, v in gp_latent_metrics_test.items():
    print(f"{k}: {v:.4f}")

# Save variables for later comparison.
gp_latent_model = gp_latent
y_pred_gp_latent_train = y_pred_train
y_pred_gp_latent_test = y_pred_test
y_std_gp_latent_test = y_std_test


GP-on-latent model using active spatial split: train=1748, test=254
Fitted latent-space kernel: 0.774**2 * Matern(length_scale=[1.34, 9.87, 100, 11.5, 100, 4.01, 16.5, 100, 7.79], nu=1.5) + WhiteKernel(noise_level=0.596)

GP-latent model TRAIN results:
CorrCoef: 0.6513
MAE: 12.2929
RMSE: 21.2844
R2: 0.4233

GP-latent model TEST results:
CorrCoef: 0.5375
MAE: 14.2882
RMSE: 19.4498
R2: 0.0138

Average predictive std on test: 21.9880

GP-latent model TEST evaluation metrics:
MAE: 14.2882
RMSE: 19.4498
Bias: 2.1242
sMAPE: 0.3782
R2: 0.0138
Correlation: 0.5375
Moran_I: 0.2953
Moran_p: 0.0010


## Model summary

This section collects the fitted models into one comparison table so you can quickly scan predictive performance and spatial residual behavior.

Rows are grouped by the evaluation split they used. The active spatial split is the most conservative comparison for the spatial models, while the earlier random split rows are kept for reference.


In [86]:
import pandas as pd
import numpy as np

# Build a compact comparison table from the stored model outputs.
# Only use train metrics when the notebook stores model-specific train results.

summary_rows = []

def add_row(name, train_metrics=None, test_metrics=None, extra=None):
    row = {"Model": name}
    if train_metrics is not None:
        row.update({f"Train_{k}": v for k, v in train_metrics.items()})
    if test_metrics is not None:
        row.update({f"Test_{k}": v for k, v in test_metrics.items()})
    if extra is not None:
        row.update(extra)
    summary_rows.append(row)

# Helper to convert metric dictionaries into a consistent subset.
def pick_metrics(metrics_dict, keys=("R2", "RMSE", "MAE", "Correlation", "Moran_I", "Moran_p", "Bias", "sMAPE")):
    if metrics_dict is None:
        return None
    return {k: metrics_dict.get(k, np.nan) for k in keys}

# 1) Ridge baseline, if present
if "y_true" in globals() and "preds" in globals():
    baseline_test_metrics = evaluate_model(y_true, preds)
    add_row(
        "Ridge baseline",
        test_metrics=pick_metrics(baseline_test_metrics),
        extra={"Split": "original random split"},
    )

# 2) Exact Bayesian linear regression
if "y_true" in globals() and "preds_exact" in globals():
    exact_test_metrics = evaluate_model(y_true, preds_exact)
    add_row(
        "Exact Bayesian linear",
        test_metrics=pick_metrics(exact_test_metrics),
        extra={"Split": "original random split"},
    )

# 3) Spatial-lag exact Bayesian linear regression
if "y_true_spatial" in globals() and "preds_spatial_exact" in globals():
    spatial_exact_metrics = evaluate_model(y_true_spatial, preds_spatial_exact, coords=df.iloc[ix_test][["x", "y"]].to_numpy())
    add_row(
        "Spatial-lag exact Bayesian",
        test_metrics=pick_metrics(spatial_exact_metrics),
        extra={"Split": "original random split"},
    )

# 4) SAR train/test metrics
if "sar_metrics_train" in globals() and "sar_metrics" in globals():
    add_row(
        "SAR (target-lag)",
        train_metrics=pick_metrics(sar_metrics_train),
        test_metrics=pick_metrics(sar_metrics),
        extra={"Split": "active spatial split" if "ix_train_active" in globals() else "original random split"},
    )

# 5) SAR sensitivity table (already holds multiple k values)
if "sar_k_comparison" in globals() and len(sar_k_comparison) > 0:
    best_k_row = sar_k_comparison.sort_values(["R2_test_raw", "Corr_test"], ascending=False).iloc[0]
    add_row(
        "SAR sensitivity (best k)",
        train_metrics={
            "R2": best_k_row.get("R2_train", np.nan),
            "RMSE": np.nan,
            "MAE": np.nan,
            "Correlation": np.nan,
            "Moran_I": np.nan,
            "Moran_p": np.nan,
            "Bias": np.nan,
            "sMAPE": np.nan,
        },
        test_metrics={
            "R2": best_k_row.get("R2_test_raw", np.nan),
            "RMSE": best_k_row.get("RMSE_test", np.nan),
            "MAE": best_k_row.get("MAE_test", np.nan),
            "Correlation": best_k_row.get("Corr_test", np.nan),
            "Moran_I": best_k_row.get("test_Moran_I", np.nan),
            "Moran_p": best_k_row.get("test_Moran_p", np.nan),
            "Bias": best_k_row.get("test_Bias", np.nan),
            "sMAPE": best_k_row.get("test_sMAPE", np.nan),
        },
        extra={"Split": "active spatial split", "k": int(best_k_row["k"])},
    )

# 6) Hierarchical model
if "hier_metrics" in globals() and "y_pred_hier_test" in globals() and "y_test_h" in globals():
    add_row(
        "Hierarchical (groups + lag)",
        train_metrics=pick_metrics(hier_train_metrics) if "hier_train_metrics" in globals() else None,
        test_metrics=pick_metrics(hier_metrics),
        extra={"Split": "active spatial split" if "ix_train_active" in globals() else "original random split"},
    )

# 7) GP-augmented hierarchical model
if "gp_hier_metrics_test" in globals() and "y_pred_gp_hier_test" in globals() and "y_sar_test" in globals():
    add_row(
        "GP-hierarchical",
        train_metrics=pick_metrics(gp_hier_metrics) if "gp_hier_metrics" in globals() else None,
        test_metrics=pick_metrics(gp_hier_metrics_test),
        extra={"Split": "active spatial split" if "ix_train_active" in globals() else "original random split"},
    )

# 8) PCA latent model
if "latent_metrics_test" in globals():
    add_row(
        "Latent PCA + Ridge",
        test_metrics=pick_metrics(latent_metrics_test),
        extra={"Split": "active spatial split" if "ix_train_active" in globals() else "original random split", "q": int(z_train.shape[1]) if "z_train" in globals() else np.nan},
    )

# 9) PPCA latent model
if "ppca_metrics_test" in globals():
    add_row(
        "PPCA + Ridge",
        test_metrics=pick_metrics(ppca_metrics_test),
        extra={"Split": "active spatial split" if "ix_train_active" in globals() else "original random split", "q": int(latent_dim) if "latent_dim" in globals() else np.nan},
    )

# 10) GP latent model
if "gp_latent_metrics_test" in globals():
    add_row(
        "GP on latent state",
        test_metrics=pick_metrics(gp_latent_metrics_test),
        extra={"Split": "active spatial split" if "ix_train_active" in globals() else "original random split", "Latent_q": int(z_train_ppca.shape[1]) if "z_train_ppca" in globals() else np.nan},
    )

summary_df = pd.DataFrame(summary_rows)

if summary_df.empty:
    print("No model summary rows were found. Run the model cells first.")
else:
    # Prefer test R2 for ranking, then test correlation.
    if "Test_R2" in summary_df.columns:
        summary_df = summary_df.sort_values(["Test_R2", "Test_Correlation"], ascending=False, na_position="last")
    display_cols = [c for c in ["Model", "Split", "k", "q", "Latent_q", "Train_R2", "Test_R2", "Train_RMSE", "Test_RMSE", "Train_MAE", "Test_MAE", "Train_Correlation", "Test_Correlation", "Train_Moran_I", "Test_Moran_I", "Train_Moran_p", "Test_Moran_p"] if c in summary_df.columns]
    print("Model comparison overview:")
    display(summary_df[display_cols].round(4))

    best_row = summary_df.iloc[0]
    print("\nBest by Test_R2 / Test_Correlation:")
    print(f"  {best_row['Model']}")
    if 'Test_R2' in best_row:
        print(f"  Test R2: {best_row.get('Test_R2', np.nan):.4f}")
    if 'Test_Correlation' in best_row:
        print(f"  Test Correlation: {best_row.get('Test_Correlation', np.nan):.4f}")


Model comparison overview:


c:\Users\rasmu\miniconda3\envs\mbml\Lib\site-packages\libpysal\weights\distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 4 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


,Model,Split,k,q,Latent_q,Train_R2,Test_R2,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE,Train_Correlation,Test_Correlation,Train_Moran_I,Test_Moran_I,Train_Moran_p,Test_Moran_p
6,GP-hierarchical,active spatial split,NaN,NaN,NaN,1.0000,0.4012,0.0000,23.5592,0.0000,12.1891,1.0000,0.6357,-0.0145,0.3125,0.001,0.001
1,Exact Bayesian linear,original random split,NaN,NaN,NaN,NaN,0.2956,NaN,25.5534,NaN,13.5417,NaN,0.5479,NaN,NaN,NaN,NaN
2,Spatial-lag exact Bayesian,original random split,NaN,NaN,NaN,NaN,0.2775,NaN,25.8790,NaN,13.5251,NaN,0.5323,NaN,0.3886,NaN,0.001
8,PPCA + Ridge,active spatial split,NaN,9.0,NaN,NaN,0.0825,NaN,18.7604,NaN,14.6020,NaN,0.5014,NaN,0.3096,NaN,0.001
7,Latent PCA + Ridge,active spatial split,NaN,9.0,NaN,NaN,0.0500,NaN,19.0899,NaN,15.0529,NaN,0.5060,NaN,0.3443,NaN,0.001
9,GP on latent state,active spatial split,NaN,NaN,9.0,NaN,0.0138,NaN,19.4498,NaN,14.2882,NaN,0.5375,NaN,0.2953,NaN,0.001
3,SAR (target-lag),active spatial split,NaN,NaN,NaN,0.4030,-0.1655,21.6563,21.1441,12.4628,16.4039,0.6348,0.5299,0.0442,0.3124,0.001,0.001
4,SAR sensitivity (best k),active spatial split,64.0,NaN,NaN,0.4068,-0.1662,NaN,21.1504,NaN,16.3796,NaN,0.5307,NaN,0.2941,NaN,0.001
5,Hierarchical (groups + lag),active spatial split,NaN,NaN,NaN,0.4764,-0.2782,20.2799,22.1428,11.9907,17.3193,0.6902,0.5220,NaN,0.3981,NaN,0.001
0,Ridge baseline,original random split,NaN,NaN,NaN,NaN,-10.0362,NaN,101.1440,NaN,81.0448,NaN,0.0989,NaN,NaN,NaN,NaN



Best by Test_R2 / Test_Correlation:
  GP-hierarchical
  Test R2: 0.4012
  Test Correlation: 0.6357


## 2.3 Pyro: Heteroscedastic regression

Ok, let us now assume again that the Gaussian likelihood was indeed the most appropriate choice. In many problems of interest, it is often the case that constant observation variance ($\sigma^2$) is too limiting or inadequate. We can relax this assumption by considering heteroscedastic models, in which the observation variance is assumed to be non-constant and dependent on any other variables. In this particular case, we shall assume that the observation variance is also linearly dependent on the inputs $\textbf{x}$. 

Lets implement this model in Pyro (check the lecture slides, if necessary)!

In [62]:
def heteroscedastic_model(X, obs=None):
    alpha_mu = pyro.sample("alpha_mu", dist.Normal(0., 1.))                 # Prior for the bias/intercept of the mean
    beta_mu  = pyro.sample("beta_mu", dist.Normal(torch.zeros(X.shape[1]), 
                                               torch.ones(X.shape[1])).to_event())     # Priors for the regression coeffcients of the mean
    alpha_v = pyro.sample("alpha_v", dist.Normal(0., 1.))                   # Prior for the bias/intercept of the variance
    beta_v  = pyro.sample("beta_v", dist.Normal(torch.zeros(X.shape[1]), 
                                               torch.ones(X.shape[1])).to_event())     # Priors for the regression coeffcients of the variance
    
    with pyro.plate("data"):
        y = pyro.sample("y", dist.Normal(alpha_mu + X.matmul(beta_mu), torch.exp(alpha_v + X.matmul(beta_v))), obs=obs)
        
    return y

Once you finished coding the model, it is time to run inference on it. Feel free to play with the hyper-parameters of the optimization (i.e. ```n_steps```, learning rate ```lr```, etc.):

In [63]:
# Prepare data for Pyro model
X_train_torch = torch.tensor(X_train).float()
y_train_torch = torch.tensor(y_train).float()

In [64]:
# Define guide function
guide = AutoMultivariateNormal(heteroscedastic_model)

# Reset parameter values
pyro.clear_param_store()

# Define the number of optimization steps
n_steps = 8000

# Setup the optimizer
adam_params = {"lr": 0.001} # learning rate (lr) of optimizer
optimizer = ClippedAdam(adam_params)

# Setup the inference algorithm
elbo = Trace_ELBO(num_particles=1)
svi = SVI(heteroscedastic_model, guide, optimizer, loss=elbo)

# Do gradient steps
for step in range(n_steps):
    elbo = svi.step(X_train_torch, y_train_torch)
    if step % 200 == 0:
        print("[%d] ELBO: %.1f" % (step, elbo))

[0] ELBO: 6230.4
[200] ELBO: 2484.8
[400] ELBO: 2073.7
[600] ELBO: 1611.2
[800] ELBO: 1629.0
[1000] ELBO: 1453.1
[1200] ELBO: 1331.3
[1400] ELBO: 1480.0
[1600] ELBO: 1299.5
[1800] ELBO: 1317.5
[2000] ELBO: 1308.0
[2200] ELBO: 1324.8
[2400] ELBO: 1313.9
[2600] ELBO: 1289.5
[2800] ELBO: 1247.5
[3000] ELBO: 1257.0
[3200] ELBO: 1321.2
[3400] ELBO: 1265.8
[3600] ELBO: 1239.5
[3800] ELBO: 1260.2
[4000] ELBO: 1242.4
[4200] ELBO: 1223.0
[4400] ELBO: 1240.0
[4600] ELBO: 1236.1
[4800] ELBO: 1230.0
[5000] ELBO: 1227.2
[5200] ELBO: 1227.6
[5400] ELBO: 1224.4
[5600] ELBO: 1232.3
[5800] ELBO: 1225.7
[6000] ELBO: 1219.4
[6200] ELBO: 1218.8
[6400] ELBO: 1214.0
[6600] ELBO: 1220.2
[6800] ELBO: 1221.6
[7000] ELBO: 1210.5
[7200] ELBO: 1226.3
[7400] ELBO: 1224.1
[7600] ELBO: 1214.2
[7800] ELBO: 1212.4


We can now extract the samples from the posterior distribution and use them to make predictions for the test set:

In [212]:
from pyro.infer import Predictive

predictive = Predictive(heteroscedastic_model, guide=guide, num_samples=1000,
                        return_sites=("alpha_mu", "beta_mu", "alpha_v", "beta_v"))
samples = predictive(X_train_torch, y_train_torch)

In [213]:
alpha_samples = samples["alpha_mu"].detach().numpy()
beta_samples = samples["beta_mu"].detach().numpy()
y_hat = np.mean(alpha_samples.T + np.dot(X_test, beta_samples[:,0].T), axis=1)

# convert back to the original scale
preds = y_hat * y_std + y_mean
y_true = y_test * y_std + y_mean

corr, mae, rae, rmse, r2 = compute_error(y_true, preds)
print("CorrCoef: %.3f\nMAE: %.3f\nRMSE: %.3f\nR2: %.3f" % (corr, mae, rmse, r2))

metrics = evaluate_model(y_true, preds)
print("\nHeteroscedastic New Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

CorrCoef: 0.364
MAE: 16.452
RMSE: 29.440
R2: 0.065

Heteroscedastic New Evaluation Metrics:
MAE: 16.4520
RMSE: 29.4400
Bias: -2.9422
sMAPE: 0.4587
R2: 0.0650
Correlation: 0.3644


In [67]:
alpha_v_samples = samples["alpha_v"].detach().numpy()
beta_v_samples = samples["beta_v"].detach().numpy()
sigma_hat = np.mean(np.exp(alpha_v_samples.T + np.dot(X_test, beta_v_samples[:,0].T)), axis=1)

In [68]:
np.set_printoptions(precision=3)
print(sigma_hat[:10])

[0.398 0.477 0.326 0.337 0.387 0.4   1.341 0.368 0.354 0.555]


Notice how it changes over time. We can use these values to estimate prediction intervals (e.g. 95% prediction intervals) that are time-varying. Can you think of real-world problems where this information would be useful?